# ISL caption, pose and target audit

This private Kaggle notebook checks the V5 fit and inner-stop caption-to-pose joins, source/alias and split-role boundaries, preprocessing metadata, duplicate-caption structure and reference-only support for the previously fixed 985-query cohort. It performs no model training and writes one aggregate JSON file. Raw clips are not opened. The reused 498-row development cohort contributes only metadata-based role-overlap and caption-duplicate counts. Its pose arrays are never loaded, and no pose metrics or support strata are computed for it; those metadata results are not used to tune or evaluate a model.

Use the approved V2 pose cache and V5 role/artifact inputs, enable Internet for the pinned public MiniLM encoder, keep the notebook Private, and select T4×2. iSign derived data remains private and subject to its research/noncommercial license.


In [ ]:
# Environment setup: consumes mounted artifacts and cached poses; never downloads pose data.
import gc, hashlib, json, math, os, random, shutil, sys, time, zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

try:
    from transformers import AutoModel, AutoTokenizer
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers'])
    from transformers import AutoModel, AutoTokenizer

SEED = 17
TMAX, POINTS, COORDS = 48, 75, 3
POSE_DIM = POINTS * COORDS
LMAX_TOKENS = 128
BATCH_SIZE, EVAL_BATCH = 16, 16
MAX_EPOCHS, PATIENCE = 25, 5
VELOCITY_WEIGHT, PHASE_WEIGHT, DYNAMIC_WEIGHT = 1.0, 0.25, 1.0
RANK_WEIGHT, RANK_MARGIN = 0.20, 0.01
NEAR_DUPLICATE_COSINE_THRESHOLD = 0.92
MIN_ANCHOR_CONF = 0.15
BONE_MIN_EDGE_SUPPORT = 100
BONE_MIN_CLIP_GROUP_SUPPORT = 96
BONE_REFERENCE_P95_MIN = 1e-3
GROUPS = {'body': (0, 33, 0.20), 'left_hand': (33, 54, 0.40), 'right_hand': (54, 75, 0.40)}
TEXT_ENCODER_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
TEXT_ENCODER_REVISION = '1110a243fdf4706b3f48f1d95db1a4f5529b4d41'
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, 'Select Kaggle GPU T4 x2 before running.'
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
device = torch.device('cuda:0')
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)
if hasattr(torch.backends.cuda, 'enable_cudnn_sdp'):
    torch.backends.cuda.enable_cudnn_sdp(False)
torch.backends.cuda.matmul.allow_tf32 = False
ATTENTION_BACKEND_CONFIG = {
    'scaled_dot_product_attention': 'math_only', 'flash': torch.backends.cuda.flash_sdp_enabled(),
    'memory_efficient': torch.backends.cuda.mem_efficient_sdp_enabled(),
    'math': torch.backends.cuda.math_sdp_enabled(),
    'cudnn': torch.backends.cuda.cudnn_sdp_enabled() if hasattr(torch.backends.cuda, 'cudnn_sdp_enabled') else None,
    'tf32': torch.backends.cuda.matmul.allow_tf32, 'model_dtype': 'float32',
}
assert ATTENTION_BACKEND_CONFIG['math'] and not ATTENTION_BACKEND_CONFIG['flash'] and not ATTENTION_BACKEND_CONFIG['memory_efficient']
BUILDER_SHA256 = 'E70B96EBC07C7EDCBAECBDF44DF6C2907ED02792317BE898AD9A98B3C7F2087A'
print('V9 builder SHA256:', BUILDER_SHA256)
print('V9 environment:', torch.__version__, '| GPUs:', GPU_NAMES, '| device:', device)
print('V9 attention backend and precision:', ATTENTION_BACKEND_CONFIG)


In [ ]:
extracted_root = None
try:
    # Load the exact V5 cohort/checkpoint and rebuild only from attached raw caches.
    V5_MANIFEST_NAME = 'manifest_minilm_position_motion_v5.json'
    manifest_paths = sorted(INPUT_ROOT.rglob(V5_MANIFEST_NAME))
    if not manifest_paths:
        artifact_zips = sorted(INPUT_ROOT.rglob('hackcessible_isl_v5_artifacts.zip'))
        if len(artifact_zips) != 1:
            raise RuntimeError(
                f'Expected the V5 artifact tree or exactly one hackcessible_isl_v5_artifacts.zip; found {len(artifact_zips)} ZIPs.'
            )
        artifact_sha = hashlib.sha256(artifact_zips[0].read_bytes()).hexdigest().upper()
        extracted_root = WORK_ROOT / f'v5_artifacts_{artifact_sha[:12]}'
        extracted_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(artifact_zips[0], 'r') as archive:
            members = archive.infolist()
            for member in members:
                member_path = Path(member.filename)
                if member_path.is_absolute() or '..' in member_path.parts:
                    raise RuntimeError('Attached V5 artifact ZIP contains an unsafe path.')
            archive.extractall(extracted_root)
        manifest_paths = sorted(extracted_root.rglob(V5_MANIFEST_NAME))
        print('Extracted attached V5 artifact ZIP:', artifact_zips[0], '| SHA256:', artifact_sha)
    if len(manifest_paths) != 1:
        raise RuntimeError(
            f'Expected exactly one attached V5 artifact manifest named {V5_MANIFEST_NAME}; '
            f'found {len(manifest_paths)}. Attach the completed V5 artifact dataset.'
        )
    V5_MANIFEST_PATH = manifest_paths[0]
    V5_ROOT = V5_MANIFEST_PATH.parent
    manifest = json.loads(V5_MANIFEST_PATH.read_text(encoding='utf-8'))
    V5_METRICS_PATH = V5_ROOT / 'metrics_summary.json'
    V5_SELECTION_PATH = V5_ROOT / 'selection_manifest_predecode.json'
    if not V5_METRICS_PATH.is_file() or not V5_SELECTION_PATH.is_file():
        raise RuntimeError('V5 metrics_summary.json and selection_manifest_predecode.json are required for raw-cache coverage and exclusion audits.')
    v5_metrics = json.loads(V5_METRICS_PATH.read_text(encoding='utf-8'))
    V5_MANIFEST_SHA256 = hashlib.sha256(V5_MANIFEST_PATH.read_bytes()).hexdigest().upper()
    V5_METRICS_SHA256 = hashlib.sha256(V5_METRICS_PATH.read_bytes()).hexdigest().upper()
    v5_selection = json.loads(V5_SELECTION_PATH.read_text(encoding='utf-8'))
    fresh_train_selected_records = list(v5_selection['fresh_train'])
    fresh_test_selected_records = list(v5_selection['fresh_test'])
    fresh_train_selected_uids = [str(row['uid']) for row in fresh_train_selected_records]
    fresh_test_selected_uids = [str(row['uid']) for row in fresh_test_selected_records]
    fresh_selected_uids = fresh_train_selected_uids + fresh_test_selected_uids
    fresh_train_selected_uid_set = set(fresh_train_selected_uids)
    fresh_test_selected_uid_set = set(fresh_test_selected_uids)
    assert len(fresh_train_selected_uids) == int(v5_metrics['fresh_train_rows_selected']) == 8000
    assert len(fresh_test_selected_uids) == int(v5_metrics['fresh_test_rows_selected']) == 500
    assert len(set(fresh_selected_uids)) == len(fresh_selected_uids) == 8500, 'V5 predecode selection must contain exactly 8,500 unique fresh UIDs.'
    assert float(v5_selection.get('minimum_fresh_coverage', 0.0)) >= 0.98
    fresh_selected_uid_set = set(fresh_selected_uids)
    v5_exclusions = list(v5_metrics.get('load_and_canonicalization_exclusions', []))
    fresh_decode_failure_uids = {
        str(row['uid']) for row in v5_exclusions
        if str(row.get('reason', '')).startswith('fresh_pose_decode_error:')
    }
    fresh_selection_by_uid = {str(row['uid']): row for row in fresh_train_selected_records + fresh_test_selected_records}
    fresh_exclusion_by_uid = {str(row['uid']): row for row in v5_exclusions if str(row.get('uid')) in fresh_selected_uid_set}
    assert fresh_decode_failure_uids.issubset(fresh_selected_uid_set), 'A logged fresh decode failure is outside the fixed predecode roster.'
    v5_exclusion_anchor_accounting = []
    for row in v5_exclusions:
        frames = int(row.get('resampled_frame_count') or TMAX)
        direct = int(row.get('direct_anchor_frames') or 0)
        fallback = int(row.get('fallback_anchor_frames') or 0)
        interpolated = int(row.get('interpolated_anchor_frames') or 0)
        unresolved = int(row.get('unresolved_anchor_frames') or 0)
        reason = str(row.get('reason', ''))
        rejected = row.get('anchor_mode') == 'unusable' or 'fewer_than_two_reliable' in reason or reason.startswith('fresh_pose_decode_error:') or reason.startswith('selected_v2_pose_missing') or reason.startswith('selected_v2_cache_error:')
        record = dict(row)
        record.update({
            'observed_anchor_frames': direct + fallback,
            'canonicalized_output_frames': 0 if rejected else max(0, frames - unresolved),
            'clip_rejected': bool(rejected),
            'logged_unresolved_frames_semantics': 'clip-level rejected/unavailable coverage; may overlap observed direct/fallback anchor counts' if rejected else 'unresolved frames in accepted clip',
            'anchor_counts_form_disjoint_frame_partition': not rejected,
        })
        if rejected:
            assert unresolved == frames, 'Rejected-anchor accounting invariant failed.'
            assert direct + fallback == int(row.get('total_anchor_frames') or 0), 'Rejected-anchor count reconciliation failed.'
        else:
            assert direct + fallback + interpolated + unresolved == frames, 'Accepted-anchor count partition failed.'
        v5_exclusion_anchor_accounting.append(record)
    assert manifest.get('model_arm') == 'minilm_position_motion_v5', f'Unexpected V5 model arm: {manifest.get("model_arm")}'
    assert manifest.get('coordinate_space', '').startswith('canonical nose/shoulder-centered pose')
    assert manifest.get('topology', {}).get('components') == [
        {'name': 'body', 'start': 0, 'count': 33},
        {'name': 'left_hand', 'start': 33, 'count': 21},
        {'name': 'right_hand', 'start': 54, 'count': 21},
    ]
    split = manifest['split']
    train_records = list(split['train']); test_records = list(split['validation'])
    train_by_uid = {str(row['uid']): row for row in train_records}
    test_by_uid = {str(row['uid']): row for row in test_records}
    assert len(train_by_uid) == len(train_records) and len(test_by_uid) == len(test_records)
    fit_uids = [str(uid) for uid in manifest['training_uids_used']]
    early_uids = [str(uid) for uid in manifest['early_stop_uids']]
    test_uids = [str(row['uid']) for row in test_records]
    assert len(set(fit_uids)) == len(fit_uids) and len(set(early_uids)) == len(early_uids)
    assert set(fit_uids).isdisjoint(early_uids) and set(fit_uids).isdisjoint(test_uids) and set(early_uids).isdisjoint(test_uids)
    assert set(fit_uids + early_uids).issubset(train_by_uid)
    assert set(test_uids) == set(test_by_uid)
    sample_by_uid = {}  # External development cohort is metadata-only.
    fixed_set = manifest['fixed_unseen_set']
    fixed_uids = list(map(str, fixed_set['uids']))
    fixed_effective = list(map(str, fixed_set.get('effective_uids', [])))
    fixed_missing = list(map(str, fixed_set.get('missing_uids', [])))
    assert len(fixed_uids) == int(fixed_set['count']) and len(set(fixed_uids)) == len(fixed_uids)
    assert set(fixed_effective).isdisjoint(fixed_missing) and set(fixed_effective) | set(fixed_missing) == set(fixed_uids)
    assert set(fixed_effective).issubset(test_uids)
    assert manifest['normalizer']['fit_uid_sha256'] == hashlib.sha256('\n'.join(sorted(fit_uids)).encode()).hexdigest()
    FIT_UID_ORDER_SHA256 = hashlib.sha256('\n'.join(fit_uids).encode()).hexdigest()
    TEST_UID_ORDER_SHA256 = hashlib.sha256('\n'.join(test_uids).encode()).hexdigest()
    print('Frozen V5 roles:', len(fit_uids), 'fit /', len(early_uids), 'inner stop /', len(test_uids), 'paired test.')

    def caption_key(row):
        return str(row.get('normalized_caption_key') or ' '.join(str(row.get('text', '')).lower().split()))
    fit_caption_keys = {caption_key(train_by_uid[uid]) for uid in fit_uids}
    stop_caption_keys = {caption_key(train_by_uid[uid]) for uid in early_uids}
    test_caption_keys = {caption_key(row) for row in test_records}
    assert fit_caption_keys.isdisjoint(stop_caption_keys | test_caption_keys)
    assert stop_caption_keys.isdisjoint(test_caption_keys)
    rank_role_uids = fit_uids + early_uids
    assert all(train_by_uid[uid].get('source_video_cluster') and train_by_uid[uid].get('source_video_id') for uid in rank_role_uids)
    assert all(isinstance(train_by_uid[uid].get('source_aliases'), list) for uid in rank_role_uids)
    SOURCE_CLUSTER_COUNT = len({str(train_by_uid[uid]['source_video_cluster']) for uid in rank_role_uids})
    def role_source_aliases(uids, rows):
        aliases = set()
        for uid in uids:
            row = rows[uid]
            aliases.update(str(value) for value in row.get('source_aliases', []) if value)
            aliases.update(str(value) for value in (row.get('source_video_cluster'), row.get('source_video_id')) if value)
        return aliases
    fit_source_aliases = role_source_aliases(fit_uids, train_by_uid)
    stop_source_aliases = role_source_aliases(early_uids, train_by_uid)
    test_source_aliases = role_source_aliases(test_uids, test_by_uid)
    assert fit_source_aliases.isdisjoint(stop_source_aliases), 'Fit and inner-stop source clusters/aliases overlap.'
    assert fit_source_aliases.isdisjoint(test_source_aliases), 'Fit and development source clusters/aliases overlap.'
    assert stop_source_aliases.isdisjoint(test_source_aliases), 'Inner-stop and development source clusters/aliases overlap.'
    assert len({str(test_by_uid[uid].get('source_video_cluster')) for uid in test_uids}) == len(test_uids), \
        'Development bootstrap assumes one row per source-video cluster.'
    SPLIT_ROLE_SHA256 = hashlib.sha256(json.dumps({
        'fit': hashlib.sha256('\n'.join(sorted(fit_uids)).encode()).hexdigest(),
        'inner_stop': hashlib.sha256('\n'.join(sorted(early_uids)).encode()).hexdigest(),
        'development_only': hashlib.sha256('\n'.join(sorted(test_uids)).encode()).hexdigest(),
    }, sort_keys=True, separators=(',', ':')).encode()).hexdigest().upper()
    print('Caption/source ranking metadata:', len(rank_role_uids), 'fit+stop rows;', SOURCE_CLUSTER_COUNT, 'source clusters; aliases available for all rows.')
    print('Explicit disjoint fit, inner-stop and development-only roles verified.')
    v5_unassigned_train_uids = sorted(set(train_by_uid) - set(fit_uids) - set(early_uids))
    v5_unassigned_train_rows = []
    for uid in v5_unassigned_train_uids:
        row = train_by_uid[uid]
        key = caption_key(row)
        reason = 'excluded_from_fit_by_normalized_caption_overlap'
        v5_unassigned_train_rows.append({'uid': uid, 'text': row.get('text'), 'reason': reason})

    checkpoint_paths = sorted(V5_ROOT.rglob('minilm_position_motion_v5.pt'))
    if len(checkpoint_paths) != 1:
        raise RuntimeError(f'Expected one V5 checkpoint under {V5_ROOT}; found {len(checkpoint_paths)}.')
    V5_CHECKPOINT_PATH = checkpoint_paths[0]
    V5_CHECKPOINT_SHA256 = hashlib.sha256(V5_CHECKPOINT_PATH.read_bytes()).hexdigest().upper()
    v5_checkpoint = torch.load(V5_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    POINT_INDICES = np.asarray(v5_checkpoint['point_indices'], dtype=np.int64)

    normalizer_path = (V5_ROOT / manifest['normalizer']['path']).resolve()
    assert normalizer_path.is_file(), f'V5 fit-only normalizer is missing: {normalizer_path}'
    with np.load(normalizer_path) as saved:
        v5_mean = np.asarray(saved[manifest['normalizer'].get('mean_key', 'mean')], dtype=np.float32).copy()
        v5_std = np.asarray(saved[manifest['normalizer'].get('std_key', 'std')], dtype=np.float32).copy()
    assert v5_mean.shape == v5_std.shape == (POINTS, COORDS) and np.isfinite(v5_mean).all() and np.isfinite(v5_std).all()
    assert np.all(v5_std >= 0.05)
    assert np.allclose(v5_mean, np.asarray(v5_checkpoint['normalizer_mean'], dtype=np.float32), atol=1e-7, rtol=1e-7)
    assert np.allclose(v5_std, np.asarray(v5_checkpoint['normalizer_std'], dtype=np.float32), atol=1e-7, rtol=1e-7)

    header_paths = sorted(INPUT_ROOT.rglob('onNSvHmicw0_header.json'))
    POSE_HEADER_PATH = header_paths[0] if header_paths else None
    POSE_HEADER_SHA256 = hashlib.sha256(POSE_HEADER_PATH.read_bytes()).hexdigest().upper() if POSE_HEADER_PATH else None
    POSE_HEADER = json.loads(POSE_HEADER_PATH.read_text(encoding='utf-8')) if POSE_HEADER_PATH else None

    def named_cache_dirs(name):
        return sorted({path.resolve() for path in INPUT_ROOT.rglob(name) if path.is_dir()})
    v2_cache_dirs = named_cache_dirs('isign_pose_cache_711ff3100b0f')
    v5_cache_dirs = named_cache_dirs('isign_pose_cache_v5')
    if not v2_cache_dirs or not v5_cache_dirs:
        raise RuntimeError('Required pose cache inputs are not attached.')
    cache_dirs = v2_cache_dirs + v5_cache_dirs

    cache_paths_by_uid = defaultdict(list)
    for cache_dir in cache_dirs:
        for path in cache_dir.glob('*.npz'):
            cache_paths_by_uid[path.stem].append(path)
    required_uids = set(fit_uids) | set(early_uids)
    missing_cache_uids = sorted(required_uids - set(cache_paths_by_uid))
    if missing_cache_uids:
        raise RuntimeError('Raw cache coverage is incomplete for a required fit/inner-stop role.')

    def load_raw_cache(uid):
        paths = sorted(cache_paths_by_uid[str(uid)])
        for path in paths:
            with np.load(path) as saved:
                if {'pose', 'confidence'}.issubset(saved.files):
                    pose = np.asarray(saved['pose'], dtype=np.float32).copy()
                    confidence = np.asarray(saved['confidence'], dtype=np.float32).copy()
                    if pose.shape == (TMAX, 1728) and confidence.shape == (TMAX, 576):
                        raw_count = int(np.asarray(saved['raw_frame_count']).item()) if 'raw_frame_count' in saved.files else None
                        raw_fps = float(np.asarray(saved['fps']).item()) if 'fps' in saved.files else None
                        return pose, confidence, raw_count, raw_fps
        raise RuntimeError('No valid cached pose for a required fit or inner-stop row.')

    def canonicalize_v5(uid_for_error, raw_pose, raw_confidence):
        raw_pose = np.asarray(raw_pose, dtype=np.float32).reshape(TMAX, 576, 3)
        raw_confidence = np.asarray(raw_confidence, dtype=np.float32).reshape(TMAX, 576)
        finite = np.isfinite(raw_pose).all(axis=-1)
        safe = np.nan_to_num(raw_pose, nan=0.0, posinf=0.0, neginf=0.0)
        conf = np.clip(np.nan_to_num(raw_confidence, nan=0.0, posinf=0.0, neginf=0.0), 0.0, 1.0)
        selected_pose = safe[:, POINT_INDICES, :].copy()
        selected_conf = conf[:, POINT_INDICES].copy() * finite[:, POINT_INDICES]
        center = (safe[:, 11, :] + safe[:, 12, :]) * 0.5
        nose = safe[:, 0, :]
        scale = np.linalg.norm(nose[:, :2] - center[:, :2], axis=-1)
        anchors = (conf[:, 0] >= MIN_ANCHOR_CONF) & (conf[:, 11] >= MIN_ANCHOR_CONF) & (conf[:, 12] >= MIN_ANCHOR_CONF)
        anchors &= finite[:, 0] & finite[:, 11] & finite[:, 12] & np.isfinite(scale) & (scale > 1e-4)
        valid = np.flatnonzero(anchors)
        if len(valid) < 2:
            raise RuntimeError('A required fit or inner-stop pose has too few reliable canonical anchors.')
        frame_ix = np.arange(TMAX)
        center_interp = np.stack([np.interp(frame_ix, valid, center[valid, axis]) for axis in range(3)], axis=-1)
        scale_interp = np.exp(np.interp(frame_ix, valid, np.log(scale[valid].clip(1e-4))))
        selected_conf *= np.where(anchors, 1.0, 0.5).astype(np.float32)[:, None]
        canonical = (selected_pose - center_interp[:, None, :]) / scale_interp[:, None, None]
        canonical = np.nan_to_num(canonical, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        info = {'direct_anchor_frames': int(len(valid)), 'interpolated_anchor_frames': int(TMAX-len(valid)),
                'unresolved_anchor_frames': 0, 'resampled_frame_count': TMAX}
        return canonical, selected_conf.astype(np.float32), info

    sample_uids = fit_uids + early_uids
    canonical_pose_by_uid, canonical_conf_by_uid = {}, {}
    raw_meta_by_uid, anchor_audit_by_uid = {}, {}
    for n, uid in enumerate(sample_uids, start=1):
        raw_pose, raw_conf, raw_count, raw_fps = load_raw_cache(uid)
        canonical_pose, canonical_conf, anchor_info = canonicalize_v5(uid, raw_pose, raw_conf)
        canonical_pose_by_uid[uid] = canonical_pose
        canonical_conf_by_uid[uid] = canonical_conf
        anchor_audit_by_uid[uid] = anchor_info
        rec = test_by_uid[uid] if uid in test_by_uid else train_by_uid[uid]
        raw_meta_by_uid[uid] = {'raw_frame_count': raw_count, 'fps': raw_fps, 'resampling_policy': rec.get('resampling_policy')}
        if uid in test_by_uid:
            sample = sample_by_uid[uid]
            ref_path = (V5_ROOT / sample['reference']['path']).resolve()
            with np.load(ref_path) as ref:
                expected_pose = np.asarray(ref['pose'], dtype=np.float32)
                expected_conf = np.asarray(ref['confidence'], dtype=np.float32)
            assert np.allclose(canonical_pose, expected_pose, atol=2e-5, rtol=2e-5)
            assert np.allclose(canonical_conf, expected_conf, atol=2e-6, rtol=2e-6)
        if n % 1000 == 0 or n == len(sample_uids):
            print(f'Reconstructed V5 canonical rows {n}/{len(sample_uids)}', flush=True)

    # Verify normalization match
    fit_pose = np.stack([canonical_pose_by_uid[uid] for uid in fit_uids])
    fit_conf = np.stack([canonical_conf_by_uid[uid] for uid in fit_uids])
    weights = fit_conf[..., None]
    den = weights.sum(axis=(0, 1))
    recomputed_mean = ((fit_pose * weights).sum(axis=(0, 1)) / np.maximum(den, 1e-6)).astype(np.float32)
    variance = (((fit_pose - recomputed_mean[None, None]) ** 2) * weights).sum(axis=(0, 1)) / np.maximum(den, 1e-6)
    recomputed_std = np.sqrt(np.maximum(variance, 0)).clip(0.05).astype(np.float32)
    assert np.allclose(recomputed_mean, v5_mean, atol=2e-5, rtol=2e-5)
    assert np.allclose(recomputed_std, v5_std, atol=2e-5, rtol=2e-5)
    del fit_pose, fit_conf, weights, den, variance; gc.collect()
    print('Normalized targets verified and aligned with V5!')

finally:
    if extracted_root is not None and extracted_root.exists():
        safe_extracted_root = extracted_root.resolve()
        if not safe_extracted_root.is_relative_to(WORK_ROOT.resolve()):
            raise RuntimeError('Temporary V5 artifact extraction escaped the working directory.')
        shutil.rmtree(safe_extracted_root)
        print('Removed temporary V5 artifact extraction after loading role metadata and required poses.')

In [ ]:
# Aggregate-only audit utilities. Row-level return values stay in memory.
"""Pure, aggregate-safe helpers for the private ISL pairing/target audit.

The notebook keeps role rows, captions, poses and identifiers in memory. This
module returns counts and distributions only; callers must not serialize its
row-level inputs or intermediate labels.
"""

from __future__ import annotations

import re
from collections.abc import Iterable, MutableMapping
from itertools import combinations
from typing import Hashable, Mapping, Sequence

import numpy as np


CAPTION_NORMALIZER = "v5_ascii_tokens_lower_v1"
MIN_CONFIDENCE = 0.15
TIME_BIN_NAMES = ("onset", "middle", "offset")
INTERPRETABILITY_QUERY_FRACTION = 0.80
INTERPRETABILITY_SUPPORT_FRACTION = 0.50
_CAPTION_TOKEN = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?", re.IGNORECASE)


class UnionFind:
    def __init__(self, size: int):
        self.parent = list(range(size))
        self.rank = [0] * size

    def find(self, item: int) -> int:
        while self.parent[item] != item:
            self.parent[item] = self.parent[self.parent[item]]
            item = self.parent[item]
        return item

    def union(self, left: int, right: int) -> None:
        left_root, right_root = self.find(left), self.find(right)
        if left_root == right_root:
            return
        if self.rank[left_root] < self.rank[right_root]:
            left_root, right_root = right_root, left_root
        self.parent[right_root] = left_root
        if self.rank[left_root] == self.rank[right_root]:
            self.rank[left_root] += 1


def normalize_caption(text: str) -> str:
    """Reproduce the V5 exact-caption key: lowercase ASCII word tokens."""
    return " ".join(_CAPTION_TOKEN.findall(str(text).lower()))


def record_field_presence(counts: MutableMapping[str, int], field_names: Iterable[object]) -> None:
    """Increment once per distinct field name in one source file or role row."""
    for name in {str(value) for value in field_names}:
        counts[name] = int(counts.get(name, 0)) + 1


def field_presence_report(
    counts: Mapping[str, int],
    denominator: int,
    field_names: Iterable[object],
    *,
    denominator_name: str,
    present_name: str,
    missing_name: str,
) -> dict[str, dict[str, int]]:
    """Return explicit aggregate present/missing counts for each metadata key."""
    denominator = int(denominator)
    if denominator < 0:
        raise ValueError("Field-presence denominator cannot be negative.")
    report = {}
    for field in sorted({str(value) for value in field_names}):
        present = int(counts.get(field, 0))
        if present < 0 or present > denominator:
            raise ValueError("Field-presence count must be within its denominator.")
        report[field] = {
            denominator_name: denominator,
            present_name: present,
            missing_name: denominator - present,
        }
    return report


def cache_metadata_copies_match(left: Sequence[object], right: Sequence[object]) -> bool:
    """Compare the metadata slots of two cached-pose candidates.

    Candidate positions are pose, confidence, raw frame count, FPS, timestamps,
    window keys, mirror/handedness keys, and exact NPZ key names. Pose/confidence
    arrays are compared by the caller; this helper includes exact key sets so
    aliases such as ``fps`` and ``frame_rate`` are not silently conflated.
    """
    if len(left) < 8 or len(right) < 8:
        raise ValueError("Cache candidates do not include the required metadata slots.")
    for index in (2, 3):
        if left[index] != right[index]:
            return False
    if set(map(str, left[5])) != set(map(str, right[5])):
        return False
    if set(map(str, left[6])) != set(map(str, right[6])):
        return False
    if set(map(str, left[7])) != set(map(str, right[7])):
        return False
    left_timestamps, right_timestamps = left[4], right[4]
    if left_timestamps is None or right_timestamps is None:
        return left_timestamps is None and right_timestamps is None
    return bool(np.array_equal(np.asarray(left_timestamps), np.asarray(right_timestamps)))


def aliases_for_record(record: Mapping[str, object]) -> set[str]:
    values = set()
    raw_aliases = record.get("source_aliases", ())
    if isinstance(raw_aliases, (list, tuple, set, np.ndarray)):
        values.update(str(value) for value in raw_aliases if str(value))
    for key in ("source_video_id", "source_video_cluster"):
        value = record.get(key)
        if value is not None and str(value):
            values.add(str(value))
    return values


def alias_group_labels(records: Sequence[Mapping[str, object]]) -> np.ndarray:
    """Merge rows that share any conservative source/alias value."""
    if not records:
        return np.empty(0, dtype=np.int64)
    uf = UnionFind(len(records))
    owner: dict[str, int] = {}
    for index, record in enumerate(records):
        aliases = aliases_for_record(record)
        if not aliases:
            raise ValueError("A role record lacks source/alias provenance.")
        for alias in aliases:
            if alias in owner:
                uf.union(index, owner[alias])
            else:
                owner[alias] = index
    roots: dict[int, int] = {}
    labels = np.asarray([roots.setdefault(uf.find(index), len(roots)) for index in range(len(records))], dtype=np.int64)
    labels.setflags(write=False)
    return labels


def _role_values(records: Sequence[Mapping[str, object]], key: str) -> set[str]:
    result = set()
    for record in records:
        if key == "aliases":
            result.update(aliases_for_record(record))
        elif key == "caption_keys":
            stored = record.get("normalized_caption_key")
            result.add(str(stored) if stored is not None else normalize_caption(str(record.get("text", ""))))
        else:
            value = record.get(key)
            if value is not None and str(value):
                result.add(str(value))
    return result


def audit_role_records(
    roles: Mapping[str, Sequence[Mapping[str, object]]],
) -> dict[str, object]:
    """Return role counts and pairwise leakage counts without row identifiers."""
    role_names = tuple(roles)
    role_reports: dict[str, object] = {}
    per_role_values: dict[str, dict[str, set[str]]] = {}
    for role, rows_value in roles.items():
        rows = list(rows_value)
        uids = [str(row.get("uid", "")) for row in rows]
        captions = [normalize_caption(str(row.get("text", ""))) for row in rows]
        stored_captions = [str(row.get("normalized_caption_key", "")) for row in rows]
        aliases = [aliases_for_record(row) for row in rows]
        duplicate_uid_rows = len(uids) - len(set(uids))
        empty_caption_rows = sum(not value for value in captions)
        caption_key_mismatch_rows = sum(
            bool(stored) and stored != computed
            for stored, computed in zip(stored_captions, captions)
        )
        missing_alias_rows = sum(not values for values in aliases)
        alias_labels = alias_group_labels(rows) if rows else np.empty(0, dtype=np.int64)
        role_reports[role] = {
            "rows": int(len(rows)),
            "unique_uid_count": int(len(set(uids))),
            "missing_uid_rows": int(sum(not value for value in uids)),
            "duplicate_uid_rows": int(duplicate_uid_rows),
            "empty_caption_rows": int(empty_caption_rows),
            "stored_caption_key_mismatch_rows": int(caption_key_mismatch_rows),
            "missing_source_alias_rows": int(missing_alias_rows),
            "unsupported_source_identity_rows": int(sum(not bool(row.get("source_uid_supported", True)) for row in rows)),
            "source_uid_mismatch_rows": int(sum(bool(row.get("source_uid_mismatch", False)) for row in rows)),
            "source_alias_group_count": int(len(np.unique(alias_labels))),
            "rows_in_repeated_source_alias_groups": int(np.count_nonzero(np.bincount(alias_labels)[alias_labels] > 1)) if len(alias_labels) else 0,
        }
        per_role_values[role] = {
            "uids": {value for value in uids if value},
            "aliases": set().union(*aliases) if aliases else set(),
            "caption_keys": set(captions),
            "source_video_ids": _role_values(rows, "source_video_id"),
            "source_video_clusters": _role_values(rows, "source_video_cluster"),
        }

    pairwise: dict[str, object] = {}
    for left, right in combinations(role_names, 2):
        left_values, right_values = per_role_values[left], per_role_values[right]
        pairwise[f"{left}__{right}"] = {
            "shared_uid_count": int(len(left_values["uids"] & right_values["uids"])),
            "shared_alias_value_count": int(len(left_values["aliases"] & right_values["aliases"])),
            "shared_exact_caption_key_count": int(len(left_values["caption_keys"] & right_values["caption_keys"])),
            "shared_source_video_id_count": int(len(left_values["source_video_ids"] & right_values["source_video_ids"])),
            "shared_source_video_cluster_count": int(len(left_values["source_video_clusters"] & right_values["source_video_clusters"])),
        }
    return {
        "caption_normalizer": CAPTION_NORMALIZER,
        "roles": role_reports,
        "pairwise_cross_role_intersections": pairwise,
    }


def caption_near_duplicate_components(
    caption_embeddings: np.ndarray,
    *,
    threshold: float = 0.92,
    chunk_size: int = 256,
) -> tuple[np.ndarray, np.ndarray, dict[str, object]]:
    """Cluster unique normalized captions and return direct near-pair indices.

    The returned integer indices refer only to the caller's in-memory caption
    order. They must never be placed in an exported report.
    """
    embeddings = np.asarray(caption_embeddings, dtype=np.float32)
    if embeddings.ndim != 2 or embeddings.shape[0] == 0 or not np.isfinite(embeddings).all():
        raise ValueError("Caption embeddings must be a nonempty finite 2D matrix.")
    if not -1.0 <= float(threshold) <= 1.0 or int(chunk_size) < 1:
        raise ValueError("Near-duplicate threshold or chunk size is invalid.")
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    if np.any(norms < 1e-8):
        raise ValueError("Caption embeddings must be nonzero.")
    unit = embeddings / norms
    uf = UnionFind(len(unit))
    columns = np.arange(len(unit), dtype=np.int64)
    pair_blocks = []
    for start in range(0, len(unit), int(chunk_size)):
        end = min(start + int(chunk_size), len(unit))
        similarities = unit[start:end] @ unit.T
        rows = np.arange(start, end, dtype=np.int64)[:, None]
        matches = (similarities >= float(threshold)) & (columns[None, :] > rows)
        local_pairs = np.argwhere(matches)
        if len(local_pairs):
            global_pairs = np.column_stack((local_pairs[:, 0] + start, local_pairs[:, 1])).astype(np.int64)
            pair_blocks.append(global_pairs)
            for left, right in global_pairs:
                uf.union(int(left), int(right))
    pairs = np.concatenate(pair_blocks, axis=0) if pair_blocks else np.empty((0, 2), dtype=np.int64)
    roots: dict[int, int] = {}
    labels = np.asarray([roots.setdefault(uf.find(index), len(roots)) for index in range(len(unit))], dtype=np.int64)
    labels.setflags(write=False)
    pairs.setflags(write=False)
    counts = np.bincount(labels)
    return labels, pairs, {
        "caption_count": int(len(unit)),
        "direct_near_duplicate_edge_count": int(len(pairs)),
        "semantic_component_count": int(len(counts)),
        "caption_rows_merged_by_edges": int(len(unit) - len(counts)),
        "largest_component_captions": int(counts.max()),
        "cosine_threshold": float(threshold),
    }


def normalized_time_bin_labels(frame_count: int) -> dict[str, np.ndarray]:
    """Assign frames and transition midpoints to normalized thirds."""
    frame_count = int(frame_count)
    if frame_count < 2:
        raise ValueError("At least two frames are required for temporal bins.")
    edges = np.asarray([1.0 / 3.0, 2.0 / 3.0], dtype=np.float64)
    frames = np.arange(frame_count, dtype=np.float64) / float(frame_count - 1)
    transitions = (np.arange(frame_count - 1, dtype=np.float64) + 0.5) / float(frame_count - 1)
    frame_labels = np.searchsorted(edges, frames, side="right").astype(np.int8)
    transition_labels = np.searchsorted(edges, transitions, side="right").astype(np.int8)
    frame_labels.setflags(write=False)
    transition_labels.setflags(write=False)
    return {"frames": frame_labels, "transitions": transition_labels}


def _quantile_summary(values: np.ndarray) -> dict[str, object]:
    array = np.asarray(values, dtype=np.float64)
    if not array.size:
        return {"count": 0, "p10": None, "median": None, "p90": None, "mean": None}
    return {
        "count": int(array.size),
        "p10": float(np.percentile(array, 10)),
        "median": float(np.median(array)),
        "p90": float(np.percentile(array, 90)),
        "mean": float(array.mean()),
    }


def confidence_support_report(
    confidence_by_query: np.ndarray,
    groups: Mapping[str, tuple[int, int]],
    *,
    threshold: float = MIN_CONFIDENCE,
    minimum_query_fraction: float = INTERPRETABILITY_QUERY_FRACTION,
    minimum_support_fraction: float = INTERPRETABILITY_SUPPORT_FRACTION,
) -> dict[str, object]:
    """Summarize target-only support for every query, anatomy and time bin.

    A frame counts as supported when at least half its region's joints pass the
    confidence threshold. A joint counts as supported when at least half the
    bin's frames pass. A transition cell requires support in both adjacent
    frames. Every denominator is the possible in-bin cell/frame/joint count.
    """
    confidence = np.asarray(confidence_by_query, dtype=np.float64)
    if confidence.ndim != 3 or confidence.shape[0] == 0 or confidence.shape[1] < 2:
        raise ValueError("Confidence must have shape (queries, frames, joints).")
    if not np.isfinite(np.nan_to_num(confidence, nan=0.0, posinf=0.0, neginf=0.0)).all():
        raise ValueError("Confidence array has invalid numeric values.")
    if not 0.0 <= threshold <= 1.0:
        raise ValueError("Confidence threshold must be in [0, 1].")
    confidence = np.clip(np.nan_to_num(confidence, nan=0.0, posinf=0.0, neginf=0.0), 0.0, 1.0)
    labels = normalized_time_bin_labels(confidence.shape[1])
    result: dict[str, object] = {
        "query_count": int(confidence.shape[0]),
        "confidence_threshold": float(threshold),
        "time_bins": {},
        "interpretability_gate": {
            "minimum_query_fraction": float(minimum_query_fraction),
            "minimum_per_query_cells_frames_joints_fraction": float(minimum_support_fraction),
            "frame_definition": "at least half of the region joints supported in that bin frame",
            "joint_definition": "at least half of the bin frames supported for that joint",
            "transition_definition": "both adjacent frames meet the confidence threshold for the joint",
            "cohort_filtering": False,
        },
    }
    bins_result: dict[str, object] = {}
    for region, (start, end) in groups.items():
        if start < 0 or end <= start or end > confidence.shape[2]:
            raise ValueError("Anatomical region is outside the confidence joint dimension.")
        region_result: dict[str, object] = {}
        width = end - start
        if width < 1:
            raise ValueError("Anatomical region must contain at least one joint.")
        for bin_index, bin_name in enumerate(TIME_BIN_NAMES):
            frame_ix = np.flatnonzero(labels["frames"] == bin_index)
            transition_ix = np.flatnonzero(labels["transitions"] == bin_index)
            if not len(frame_ix):
                raise ValueError("A normalized time bin has no frames.")
            cells = confidence[:, frame_ix, start:end] >= threshold
            cell_fraction = cells.mean(axis=(1, 2))
            frame_fraction = (cells.mean(axis=2) >= 0.5).mean(axis=1)
            joint_fraction = (cells.mean(axis=1) >= 0.5).mean(axis=1)
            if len(transition_ix):
                transition_cells = (
                    (confidence[:, transition_ix, start:end] >= threshold)
                    & (confidence[:, transition_ix + 1, start:end] >= threshold)
                )
                transition_fraction = transition_cells.mean(axis=(1, 2))
                supported_transition_fraction = (transition_cells.mean(axis=2) >= 0.5).mean(axis=1)
            else:
                transition_fraction = np.zeros(confidence.shape[0], dtype=np.float64)
                supported_transition_fraction = np.zeros(confidence.shape[0], dtype=np.float64)
            query_pass = (
                (cell_fraction >= minimum_support_fraction)
                & (frame_fraction >= minimum_support_fraction)
                & (joint_fraction >= minimum_support_fraction)
            )
            pass_fraction = float(query_pass.mean())
            gate_applies = region in {"left_hand", "right_hand"}
            gate_met = bool(pass_fraction >= minimum_query_fraction) if gate_applies else None
            region_result[bin_name] = {
                "possible_frames_per_query": int(len(frame_ix)),
                "possible_joints_per_query": int(width),
                "possible_cells_per_query": int(len(frame_ix) * width),
                "possible_transitions_per_query": int(len(transition_ix)),
                "possible_transition_cells_per_query": int(len(transition_ix) * width),
                "cell_support_fraction": _quantile_summary(cell_fraction),
                "supported_frame_fraction": _quantile_summary(frame_fraction),
                "supported_joint_fraction": _quantile_summary(joint_fraction),
                "transition_cell_support_fraction": _quantile_summary(transition_fraction),
                "supported_transition_fraction": _quantile_summary(supported_transition_fraction),
                "queries_meeting_each_50pct_threshold": int(query_pass.sum()),
                "query_fraction_meeting_each_50pct_threshold": pass_fraction,
                "interpretability_gate_applies": bool(gate_applies),
                "interpretability_gate_met": gate_met,
                "support_status": "interpretable_by_predeclared_gate" if gate_met is True else (
                    "support_limited" if gate_applies else "reported_no_predeclared_gate"
                ),
                "full_fixed_cohort_retained": int(confidence.shape[0]),
            }
        bins_result[region] = region_result
    result["time_bins"] = bins_result
    return result


def timestamp_grid_check(
    timestamps: Sequence[float] | np.ndarray | None,
    *,
    raw_frame_count: int | None,
    fps: float | None,
    expected_frame_count: int = 48,
) -> dict[str, object]:
    """Check a stored timestamp grid against uniform frame-index resampling.

    The check uses half of one source-frame duration as tolerance. Missing
    timestamps, raw count or FPS are reported as unavailable, never inferred.
    """
    if timestamps is None or raw_frame_count is None or fps is None:
        return {"status": "unavailable", "reason": "timestamps_raw_frame_count_or_fps_missing"}
    values = np.asarray(timestamps, dtype=np.float64)
    raw_frame_count = int(raw_frame_count)
    fps = float(fps)
    if values.ndim != 1 or not np.isfinite(values).all() or raw_frame_count < 2 or not np.isfinite(fps) or fps <= 0:
        return {"status": "invalid_metadata", "reason": "timestamp_shape_or_frame_metadata_invalid"}
    if values.size not in {int(expected_frame_count), raw_frame_count}:
        return {"status": "unavailable", "reason": "timestamp_length_not_raw_or_resampled_frame_count"}
    expected_indices = np.linspace(0.0, raw_frame_count - 1.0, int(expected_frame_count), dtype=np.float64)
    if values.size == int(expected_frame_count):
        expected = values[0] + expected_indices / fps
        actual = values
        mode = "resampled_frame_times_vs_uniform_source_frame_grid"
    else:
        expected = values[0] + np.arange(raw_frame_count, dtype=np.float64) / fps
        actual = values
        mode = "raw_frame_times_vs_fps"
    tolerance = 0.5 / fps
    max_error = float(np.max(np.abs(actual - expected)))
    return {
        "status": "passed" if max_error <= tolerance + 1e-12 else "failed",
        "timestamp_mode": mode,
        "frame_count_checked": int(len(actual)),
        "half_frame_tolerance_seconds": float(tolerance),
        "maximum_absolute_error_seconds": max_error,
    }


def confidence_weighted_pair_mse(
    left_pose: np.ndarray,
    left_confidence: np.ndarray,
    right_pose: np.ndarray,
    right_confidence: np.ndarray,
    *,
    threshold: float = MIN_CONFIDENCE,
) -> float | None:
    """Compare two normalized motions only on their shared supported cells."""
    left, right = np.asarray(left_pose, dtype=np.float64), np.asarray(right_pose, dtype=np.float64)
    left_conf = np.asarray(left_confidence, dtype=np.float64)
    right_conf = np.asarray(right_confidence, dtype=np.float64)
    if left.ndim != 3 or left.shape[-1] != 3 or left.shape != right.shape:
        raise ValueError("Motion poses must match with shape (frames, joints, 3).")
    if left_conf.shape != left.shape[:2] or right_conf.shape != left.shape[:2]:
        raise ValueError("Motion confidence arrays must align with pose frames and joints.")
    if not all(np.isfinite(value).all() for value in (left, right, left_conf, right_conf)):
        raise ValueError("Motion pair contains non-finite values.")
    mask = (left_conf >= threshold) & (right_conf >= threshold)
    if not mask.any():
        return None
    weights = np.minimum(left_conf, right_conf) * mask
    cell_error = np.mean(np.square(left - right), axis=-1)
    denominator = float(weights.sum())
    return float((cell_error * weights).sum() / denominator) if denominator > 0 else None


def paired_distance_summary(values: Sequence[float]) -> dict[str, object]:
    result = _quantile_summary(np.asarray(values, dtype=np.float64))
    result["supported_pair_count"] = int(result.pop("count"))
    return result

In [ ]:
# Aggregate audit core helpers; no dataset loading or writing.
from collections import defaultdict
from typing import Hashable, Iterable, Mapping, Sequence
import numpy as np
SEMANTIC_DUPLICATE_THRESHOLD = 0.92

class UnionFind:
    def __init__(self, size: int):
        self.parent = list(range(size))
        self.rank = [0] * size

    def find(self, item: int) -> int:
        while self.parent[item] != item:
            self.parent[item] = self.parent[self.parent[item]]
            item = self.parent[item]
        return item

    def union(self, left: int, right: int) -> None:
        left_root = self.find(left)
        right_root = self.find(right)
        if left_root == right_root:
            return
        if self.rank[left_root] < self.rank[right_root]:
            left_root, right_root = right_root, left_root
        self.parent[right_root] = left_root
        if self.rank[left_root] == self.rank[right_root]:
            self.rank[left_root] += 1

def merge_alias_groups(uids: Sequence[str], aliases_by_uid: Mapping[str, Iterable[str]]) -> dict[str, int]:
    """Merge rows connected by any source/alias value without exposing aliases."""
    uf = UnionFind(len(uids))
    owner: dict[str, int] = {}
    for index, uid in enumerate(uids):
        aliases = {str(value) for value in aliases_by_uid[uid] if str(value)}
        if not aliases:
            raise ValueError("A role row is missing source/alias provenance.")
        for alias in aliases:
            if alias in owner:
                uf.union(index, owner[alias])
            else:
                owner[alias] = index
    roots: dict[int, int] = {}
    return {uid: roots.setdefault(uf.find(index), len(roots)) for index, uid in enumerate(uids)}

def source_group_derangement(
    target_source_groups: Sequence[Hashable],
    caption_source_groups: Sequence[Hashable],
    target_caption_keys: Sequence[str],
    donor_caption_keys: Sequence[str],
    caption_embeddings: np.ndarray,
    *,
    threshold: float = SEMANTIC_DUPLICATE_THRESHOLD,
    seed: int = 2307,
    max_restarts: int = 12,
) -> np.ndarray:
    """Return a bijective source-group derangement of caption labels.

    The permutation preserves the caption marginal exactly. Every assigned
    label comes from a different source/alias group and is screened against the
    target's exact normalized caption and frozen-encoder near-duplicate score.
    """
    n = len(target_source_groups)
    if not (
        len(caption_source_groups) == n
        and len(target_caption_keys) == n
        and len(donor_caption_keys) == n
        and len(caption_embeddings) == n
    ):
        raise ValueError("Caption derangement inputs have inconsistent row counts.")
    if n < 2:
        raise ValueError("A caption derangement requires at least two rows.")
    embeddings = np.asarray(caption_embeddings, dtype=np.float32)
    if embeddings.ndim != 2 or not np.isfinite(embeddings).all():
        raise ValueError("Caption embeddings must be a finite 2D matrix.")
    norms = np.linalg.norm(embeddings, axis=1)
    if np.any(norms < 1e-8):
        raise ValueError("Caption embeddings must be nonzero.")
    embeddings = embeddings / norms[:, None]

    def valid(target_ix: int, caption_ix: int) -> bool:
        if target_source_groups[target_ix] == caption_source_groups[caption_ix]:
            return False
        if normalize_caption(target_caption_keys[target_ix]) == normalize_caption(donor_caption_keys[caption_ix]):
            return False
        similarity = float(embeddings[target_ix] @ embeddings[caption_ix])
        return similarity < threshold

    rng = np.random.default_rng(seed)
    for _ in range(max_restarts):
        permutation = rng.permutation(n)
        for _repair in range(n + 1):
            invalid = [ix for ix, donor in enumerate(permutation) if not valid(ix, int(donor))]
            if not invalid:
                if len(np.unique(permutation)) != n:
                    raise AssertionError("Caption marginal was not preserved by the derangement.")
                return permutation.astype(np.int64)
            rng.shuffle(invalid)
            changed = False
            candidate_order = rng.permutation(n)
            quick_candidates = candidate_order[: min(128, n)]
            for target_ix in invalid:
                for other_ix_value in quick_candidates:
                    other_ix = int(other_ix_value)
                    if other_ix == target_ix:
                        continue
                    donor_for_target = int(permutation[other_ix])
                    donor_for_other = int(permutation[target_ix])
                    if valid(target_ix, donor_for_target) and valid(other_ix, donor_for_other):
                        permutation[target_ix], permutation[other_ix] = donor_for_target, donor_for_other
                        changed = True
                        break
                if changed:
                    break
            if not changed:
                # Rare dense-duplicate cases may need a full scan. Keep that
                # fallback out of the common path for the 9k-row role.
                for target_ix in invalid:
                    for other_ix_value in candidate_order:
                        other_ix = int(other_ix_value)
                        if other_ix == target_ix:
                            continue
                        donor_for_target = int(permutation[other_ix])
                        donor_for_other = int(permutation[target_ix])
                        if valid(target_ix, donor_for_target) and valid(other_ix, donor_for_other):
                            permutation[target_ix], permutation[other_ix] = donor_for_target, donor_for_other
                            changed = True
                            break
                    if changed:
                        break
            if not changed:
                break
    raise ValueError("Could not construct a screened source-group caption derangement.")

def common_donor_mask(
    query_pose_aliases: Iterable[str],
    true_caption_aliases: Iterable[str],
    shuffled_caption_aliases: Iterable[str],
    true_caption_key: str,
    shuffled_caption_key: str,
    donor_aliases: Sequence[Iterable[str]],
    donor_caption_keys: Sequence[str],
    donor_embeddings: np.ndarray,
    true_caption_embedding: np.ndarray,
    shuffled_caption_embedding: np.ndarray,
    confidence_support: Sequence[bool],
    *,
    semantic_threshold: float = SEMANTIC_DUPLICATE_THRESHOLD,
) -> tuple[np.ndarray, dict[str, int]]:
    """Build one donor intersection shared by all caption/control conditions."""
    n = len(donor_aliases)
    if not (
        len(donor_caption_keys) == len(donor_embeddings) == len(confidence_support) == n
    ):
        raise ValueError("Common-pool inputs have inconsistent donor row counts.")
    true_embedding = np.asarray(true_caption_embedding, dtype=np.float32)
    shuffled_embedding = np.asarray(shuffled_caption_embedding, dtype=np.float32)
    donor_embeddings = np.asarray(donor_embeddings, dtype=np.float32)
    if donor_embeddings.ndim != 2 or true_embedding.shape != (donor_embeddings.shape[1],) or shuffled_embedding.shape != true_embedding.shape:
        raise ValueError("Common-pool text embeddings have inconsistent dimensions.")
    if not np.isfinite(donor_embeddings).all() or not np.isfinite(true_embedding).all() or not np.isfinite(shuffled_embedding).all():
        raise ValueError("Common-pool text embeddings must be finite.")
    donor_norms = np.linalg.norm(donor_embeddings, axis=1, keepdims=True)
    true_norm = float(np.linalg.norm(true_embedding))
    shuffled_norm = float(np.linalg.norm(shuffled_embedding))
    if np.any(donor_norms < 1e-8) or true_norm < 1e-8 or shuffled_norm < 1e-8:
        raise ValueError("Common-pool text embeddings must be nonzero.")
    donor_embeddings = donor_embeddings / donor_norms
    true_embedding = true_embedding / true_norm
    shuffled_embedding = shuffled_embedding / shuffled_norm
    query_alias_set = set(query_pose_aliases) | set(true_caption_aliases) | set(shuffled_caption_aliases)
    source_reject = np.asarray([bool(query_alias_set.intersection(set(aliases))) for aliases in donor_aliases], dtype=bool)
    true_key = normalize_caption(true_caption_key)
    shuffled_key = normalize_caption(shuffled_caption_key)
    exact_reject = np.asarray([
        normalize_caption(key) in {true_key, shuffled_key} for key in donor_caption_keys
    ], dtype=bool)
    true_sim = donor_embeddings @ true_embedding
    shuffled_sim = donor_embeddings @ shuffled_embedding
    semantic_reject = (true_sim >= semantic_threshold) | (shuffled_sim >= semantic_threshold)
    confidence_reject = ~np.asarray(confidence_support, dtype=bool)
    common = ~(source_reject | exact_reject | semantic_reject | confidence_reject)
    return common, {
        "candidate_donors": n,
        "same_source_or_alias_rejects": int(source_reject.sum()),
        "exact_caption_rejects": int((exact_reject & ~source_reject).sum()),
        "semantic_near_duplicate_rejects": int((semantic_reject & ~source_reject & ~exact_reject).sum()),
        "confidence_support_rejects": int((confidence_reject & ~source_reject & ~exact_reject & ~semantic_reject).sum()),
        "common_donors": int(common.sum()),
    }

def confidence_support_masks(confidence):
    confidence = np.clip(np.nan_to_num(np.asarray(confidence), nan=0.0, posinf=0.0, neginf=0.0), 0.0, 1.0)
    if confidence.ndim != 2 or confidence.shape[1] != max(end for _, end, _ in GROUPS.values()) or confidence.shape[0] < 2:
        raise ValueError('Confidence array has an unexpected shape for grouped support checks.')
    confidence = confidence >= MIN_CONFIDENCE_FOR_SUPPORT
    return {
        name: {'residual':confidence[:, start:end],
               'velocity':confidence[1:, start:end] & confidence[:-1, start:end]}
        for name, (start, end, _) in GROUPS.items()
    }

def support_dimensions(mask):
    mask = np.asarray(mask, dtype=bool)
    if mask.ndim != 2:
        raise ValueError('Role support dimensions expect a frame-by-joint mask.')
    return {'cells':int(mask.sum()), 'frames':int(mask.any(axis=1).sum()),
            'joints':int(mask.any(axis=0).sum())}

def role_group_support(confidence):
    masks = confidence_support_masks(confidence)
    summary = {}
    for name in GROUPS:
        residual = support_dimensions(masks[name]['residual'])
        velocity = support_dimensions(masks[name]['velocity'])
        residual_pass = (residual['cells'] >= MIN_RESIDUAL_SUPPORTED_CELLS
                         and residual['frames'] >= MIN_RESIDUAL_SUPPORTED_FRAMES
                         and residual['joints'] >= MIN_RESIDUAL_SUPPORTED_JOINTS)
        phase_pass = velocity['frames'] >= MIN_PHASE_SUPPORTED_TRANSITIONS
        velocity_pass = (velocity['cells'] >= MIN_VELOCITY_SUPPORTED_CELLS
                         and velocity['frames'] >= MIN_VELOCITY_SUPPORTED_TRANSITIONS
                         and velocity['joints'] >= MIN_VELOCITY_SUPPORTED_JOINTS
                         and phase_pass)
        summary[name] = {
            'residual':{**residual, 'supported':bool(residual_pass)},
            'velocity':{**velocity, 'supported':bool(velocity_pass)},
            'phase_supported_transitions':velocity['frames'],
            'phase_profile_supported':bool(phase_pass),
        }
    return summary

def stack_confidence_support_rows(confidences):
    rows = [confidence_support_masks(confidence) for confidence in confidences]
    if not rows:
        raise ValueError('At least one fit donor is required for support checks.')
    return {name: {kind: np.stack([row[name][kind] for row in rows], axis=0)
                   for kind in ('residual', 'velocity')} for name in GROUPS}

def paired_group_support(target_confidence, donor_support_bits):
    target_bits = confidence_support_masks(target_confidence)
    paired = {}
    for name in GROUPS:
        group_result = {}
        for kind, minimum_cells, minimum_frames, minimum_joints in (
            ('residual', MIN_RESIDUAL_SUPPORTED_CELLS, MIN_RESIDUAL_SUPPORTED_FRAMES, MIN_RESIDUAL_SUPPORTED_JOINTS),
            ('velocity', MIN_VELOCITY_SUPPORTED_CELLS, MIN_VELOCITY_SUPPORTED_TRANSITIONS, MIN_VELOCITY_SUPPORTED_JOINTS),
        ):
            donor_mask = donor_support_bits[name][kind]
            query_mask = target_bits[name][kind]
            if donor_mask.ndim != 3 or donor_mask.shape[1:] != query_mask.shape:
                raise ValueError('Confidence support masks have inconsistent donor/query shapes.')
            overlap = donor_mask & query_mask[None, :, :]
            cells = overlap.sum(axis=(1, 2))
            frames = overlap.any(axis=2).sum(axis=1)
            joints = overlap.any(axis=1).sum(axis=1)
            passed = (cells >= minimum_cells) & (frames >= minimum_frames) & (joints >= minimum_joints)
            if kind == 'velocity':
                phase_supported = frames >= MIN_PHASE_SUPPORTED_TRANSITIONS
                passed &= phase_supported
                group_result['phase_supported_transitions'] = frames
                group_result['phase_profile_supported'] = phase_supported
            group_result[kind + '_cells'] = cells
            group_result[kind + ('_transitions' if kind == 'velocity' else '_frames')] = frames
            group_result[kind + '_joints'] = joints
            group_result[kind] = passed
        group_result['both'] = group_result['residual'] & group_result['velocity']
        paired[name] = group_result
    all_groups = np.logical_and.reduce([paired[name]['both'] for name in GROUPS])
    return paired, all_groups, role_group_support(target_confidence)

def summarize_integer_counts(values):
    values = np.asarray(values, dtype=np.int64)
    if values.size == 0:
        return {'rows': 0, 'minimum': None, 'p10': None, 'mean': None, 'maximum': None}
    return {'rows': int(values.size), 'minimum': int(values.min()),
            'p10': float(np.percentile(values, 10)), 'mean': float(values.mean()),
            'maximum': int(values.max())}

def positive_weighted_mean(values, weights):
    values = np.asarray(values)
    expanded_weights = np.broadcast_to(np.asarray(weights), values.shape)
    weight_sum = float(expanded_weights.sum())
    if not np.isfinite(weight_sum) or weight_sum <= 0:
        raise ValueError('A weighted mean requires positive finite confidence support.')
    return float((values * expanded_weights).sum() / max(weight_sum, 1e-12))

def run_support_synthetic_tests():
    full = np.ones((16, 75), dtype=np.float32)
    full_donors = stack_confidence_support_rows([full])
    _, fully_supported, _ = paired_group_support(full, full_donors)
    assert fully_supported.tolist() == [True], 'Full confidence support should pass every group.'

    missing_left = full.copy()
    missing_left[:, 33:54] = 0
    left_donors = stack_confidence_support_rows([missing_left])
    left_pairs, left_all, _ = paired_group_support(full, left_donors)
    assert not left_pairs['left_hand']['both'][0] and not left_all[0], 'Missing left-hand support must be rejected.'

    missing_right_target = full.copy()
    missing_right_target[:, 54:75] = 0
    right_pairs, right_all, _ = paired_group_support(missing_right_target, full_donors)
    assert not right_pairs['right_hand']['both'][0] and not right_all[0], 'Missing right-hand target support must be rejected.'

    low_confidence = np.full((16, 75), 0.10, dtype=np.float32)
    low_donors = stack_confidence_support_rows([low_confidence])
    _, low_all, _ = paired_group_support(low_confidence, low_donors)
    assert not low_all[0], 'Confidence below threshold must be rejected.'

    one_frame = np.zeros((16, 75), dtype=np.float32)
    one_frame[0] = 0.9
    one_frame_donors = stack_confidence_support_rows([one_frame])
    one_frame_pairs, one_frame_all, _ = paired_group_support(one_frame, one_frame_donors)
    assert not one_frame_pairs['body']['residual'][0] and not one_frame_all[0], 'Single-frame support must be rejected.'

    one_transition = np.zeros((16, 75), dtype=np.float32)
    one_transition[[0, 2, 4, 6, 8, 10, 12, 14, 15]] = 0.9
    one_transition_donors = stack_confidence_support_rows([one_transition])
    one_transition_pairs, one_transition_all, _ = paired_group_support(one_transition, one_transition_donors)
    assert one_transition_pairs['body']['residual'][0] and not one_transition_pairs['body']['velocity'][0]
    assert not one_transition_all[0], 'One supported transition must fail the velocity and phase-profile gate.'

    minimum_valid = np.zeros((16, 75), dtype=np.float32)
    for start, end, _ in GROUPS.values():
        minimum_valid[:9, start:start+MIN_RESIDUAL_SUPPORTED_JOINTS] = 0.16
    minimum_donors = stack_confidence_support_rows([minimum_valid])
    _, minimum_all, _ = paired_group_support(minimum_valid, minimum_donors)
    assert minimum_all.tolist() == [True], 'Support at the declared cell/frame/joint minima should pass.'

    low_weight_mean = positive_weighted_mean(np.asarray([2.0, 4.0]), np.asarray([0.001, 0.001]))
    assert abs(low_weight_mean - 3.0) < 1e-9, 'Weighted mean must normalize by actual support below total weight 1.'
    return True


In [ ]:
# Pairing, source/alias, metadata and target-support audit. No model fitting.
import collections
import gc
import hashlib
import json
import math
import random
import re
from itertools import combinations

import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer

AUDIT_BUILDER_SHA256 = "E70B96EBC07C7EDCBAECBDF44DF6C2907ED02792317BE898AD9A98B3C7F2087A"
AUDIT_CORE_SHA256 = "B0057816ECC7387EF66B5693678F3A155CB2465C7D91ED3439C5ACA2F9233AE5"
CANONICALIZER_SOURCE_SHA256 = "9194C578A84C7F6C18DFD4C5DF42942B30EECECCE18A62BB91575C938CF3CEA7"
EXPECTED_V2_RUN_KEY = "711ff3100b0f"
EXPECTED_V5_CHECKPOINT_SHA256 = "CFE5CA02DD9287EE74C5C12798BC0D65F6AF87F4071754DDB40E358E0D6C98A1"
EXPECTED_V5_MANIFEST_SHA256 = "6A72903B09707E2B3EB76020C8E54CFD3EDDBEAA6098662126FABDC1BC795E69"
TEXT_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TEXT_ENCODER_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
SEMANTIC_DUPLICATE_COSINE_THRESHOLD = 0.92
MIN_CONFIDENCE_FOR_SUPPORT = 0.15
MIN_COMMON_DONORS_PER_QUERY = 100
EXPECTED_FIXED_QUERY_COUNT = 985
QUERY_SHUFFLE_SEED = 1703 + 211
GROUPS = {"body": (0, 33, 0.20), "left_hand": (33, 54, 0.40), "right_hand": (54, 75, 0.40)}
MIN_RESIDUAL_SUPPORTED_CELLS = 64
MIN_RESIDUAL_SUPPORTED_FRAMES = 8
MIN_RESIDUAL_SUPPORTED_JOINTS = 8
MIN_VELOCITY_SUPPORTED_CELLS = 64
MIN_VELOCITY_SUPPORTED_TRANSITIONS = 8
MIN_VELOCITY_SUPPORTED_JOINTS = 8
MIN_PHASE_SUPPORTED_TRANSITIONS = 8
TIME_BIN_NAMES = ("onset", "middle", "offset")

# Synthetic checks run before accessing any role data.
def _audit_synthetic_smoke():
    assert normalize_caption(" Wake UP, please! ") == "wake up please"
    bins = normalized_time_bin_labels(7)
    assert bins["frames"].tolist() == [0, 0, 1, 1, 2, 2, 2]
    assert np.bincount(bins["transitions"], minlength=3).tolist() == [2, 2, 2]
    full = confidence_support_report(
        np.ones((2, 48, 75), dtype=np.float32),
        {"body": (0, 33), "left_hand": (33, 54), "right_hand": (54, 75)},
    )
    assert full["time_bins"]["left_hand"]["onset"]["interpretability_gate_met"]
    empty = confidence_support_report(
        np.zeros((2, 48, 75), dtype=np.float32),
        {"body": (0, 33), "left_hand": (33, 54), "right_hand": (54, 75)},
    )
    assert empty["time_bins"]["right_hand"]["offset"]["support_status"] == "support_limited"
    ts = 5.0 + np.arange(5, dtype=np.float64) / 25.0
    assert timestamp_grid_check(ts, raw_frame_count=5, fps=25.0)["status"] == "passed"
    return True

assert _audit_synthetic_smoke()
assert len(fit_uids) == 9048 and len(early_uids) == 989
assert len(test_uids) == 498
assert len(fit_uids) + len(early_uids) == 10037
assert V5_MANIFEST_SHA256 == EXPECTED_V5_MANIFEST_SHA256
assert V5_CHECKPOINT_SHA256 == EXPECTED_V5_CHECKPOINT_SHA256
assert str(manifest.get("dataset", "")).startswith("Exploration-Lab/iSign")
assert int(manifest.get("resampling", {}).get("resampled_frame_count", -1)) == 48
assert manifest.get("resampling", {}).get("policy") == "uniform_linear_48_endpoints_included"
assert manifest.get("topology", {}).get("components") == [
    {"name": "body", "start": 0, "count": 33},
    {"name": "left_hand", "start": 33, "count": 21},
    {"name": "right_hand", "start": 54, "count": 21},
]
assert manifest.get("coordinate_space", "").startswith("canonical nose/shoulder-centered pose")
assert POINT_INDICES.tolist() == list(range(33)) + list(range(501, 543))

FIT_SET, STOP_SET = set(fit_uids), set(early_uids)
TRAIN_SET = set(train_by_uid)
UNASSIGNED_TRAIN_UIDS = TRAIN_SET - FIT_SET - STOP_SET
assert not FIT_SET & STOP_SET
FIT_RECORDS = [train_by_uid[uid] for uid in fit_uids]
STOP_RECORDS = [train_by_uid[uid] for uid in early_uids]
UNASSIGNED_RECORDS = [train_by_uid[uid] for uid in sorted(UNASSIGNED_TRAIN_UIDS)]
EXTERNAL_DEV_RECORDS = [test_by_uid[uid] for uid in test_uids]
AUDIT_ROLES = {
    "fit": FIT_RECORDS,
    "inner_stop": STOP_RECORDS,
    "unassigned_train_excluded": UNASSIGNED_RECORDS,
    "external_498_development_only": EXTERNAL_DEV_RECORDS,
}
ROLE_AUDIT = audit_role_records(AUDIT_ROLES)
PROBE_UIDS = fit_uids + early_uids
PROBE_RECORDS = FIT_RECORDS + STOP_RECORDS
FIT_STOP_ALIAS_GROUPS = alias_group_labels(PROBE_RECORDS)
FIT_ALIAS_GROUPS = FIT_STOP_ALIAS_GROUPS[:len(fit_uids)]
QUERY_ALIAS_GROUPS = FIT_STOP_ALIAS_GROUPS[len(fit_uids):]

def _role_overlap(left, right, metric):
    return int(ROLE_AUDIT["pairwise_cross_role_intersections"][left + "__" + right][metric])

ACTIVE_ROLE_INTEGRITY = {
    "fit_inner_stop_shared_uid_count": _role_overlap("fit", "inner_stop", "shared_uid_count"),
    "fit_inner_stop_shared_alias_value_count": _role_overlap("fit", "inner_stop", "shared_alias_value_count"),
    "fit_inner_stop_shared_exact_caption_key_count": _role_overlap("fit", "inner_stop", "shared_exact_caption_key_count"),
    "fit_external_dev_shared_uid_count": _role_overlap("fit", "external_498_development_only", "shared_uid_count"),
    "fit_external_dev_shared_alias_value_count": _role_overlap("fit", "external_498_development_only", "shared_alias_value_count"),
    "fit_external_dev_shared_exact_caption_key_count": _role_overlap("fit", "external_498_development_only", "shared_exact_caption_key_count"),
    "inner_stop_external_dev_shared_uid_count": _role_overlap("inner_stop", "external_498_development_only", "shared_uid_count"),
    "inner_stop_external_dev_shared_alias_value_count": _role_overlap("inner_stop", "external_498_development_only", "shared_alias_value_count"),
    "inner_stop_external_dev_shared_exact_caption_key_count": _role_overlap("inner_stop", "external_498_development_only", "shared_exact_caption_key_count"),
}
UNASSIGNED_EXCLUSION_AUDIT = {
    "excluded_train_rows": int(len(UNASSIGNED_RECORDS)),
    "exact_caption_key_overlap_with_fit": _role_overlap("fit", "unassigned_train_excluded", "shared_exact_caption_key_count"),
    "exact_caption_key_overlap_with_inner_stop": _role_overlap("inner_stop", "unassigned_train_excluded", "shared_exact_caption_key_count"),
    "source_alias_value_overlap_with_fit": _role_overlap("fit", "unassigned_train_excluded", "shared_alias_value_count"),
    "active_model_or_scoring_role": False,
    "pose_rows_loaded": False,
    "classification": "excluded_from_fit_by_normalized_caption_overlap",
}

ALL_ROLE_RECORDS = FIT_RECORDS + STOP_RECORDS + UNASSIGNED_RECORDS + EXTERNAL_DEV_RECORDS
ALL_ROLE_NAMES = (
    ["fit"] * len(FIT_RECORDS)
    + ["inner_stop"] * len(STOP_RECORDS)
    + ["unassigned_train_excluded"] * len(UNASSIGNED_RECORDS)
    + ["external_498_development_only"] * len(EXTERNAL_DEV_RECORDS)
)
ALL_ALIAS_GROUPS = alias_group_labels(ALL_ROLE_RECORDS)

# Audit the exact caption key stored by V5 against the V5 text-key code.
CAPTION_KEY_MISMATCH_ROWS = int(sum(
    bool(str(row.get("normalized_caption_key", "")))
    and str(row.get("normalized_caption_key")) != normalize_caption(str(row.get("text", "")))
    for row in ALL_ROLE_RECORDS
))

# Read NPZ metadata only for the 9,048 fit and 989 inner-stop rows.
# The 498 reused development cohort is not read for poses or scored.
_TIMESTAMP_KEYS = ("timestamps", "frame_timestamps", "time_stamps", "frame_times")
_WINDOW_KEYS = ("window_start", "window_end", "start_frame", "end_frame", "start_time", "end_time", "frame_start", "frame_end")
_MIRROR_KEYS = ("mirrored", "is_mirrored", "mirror", "handedness", "left_right_swapped", "side_swapped", "flipped")
_CACHE_COUNTS = collections.Counter()
_TIMESTAMP_CHECKS = collections.Counter()
_CACHE_COPY_UIDS = 0
_CACHE_COPY_CONFLICT_UIDS = set()
_CACHE_COPY_KEYSET_CONFLICT_UIDS = set()
_CACHE_COPY_INVALID = 0
_RAW_FRAME_COUNTS = []
_FPS_VALUES = []
_TIMESTAMP_PRESENT = 0
_WINDOW_PRESENT = 0
_MIRROR_PRESENT = 0
_HANDEDNESS_PRESENT = 0
_CACHE_KEY_NAMES = set()
_CACHE_KEY_FILE_PRESENCE = collections.Counter()
_CACHE_ROLE_KEY_PRESENCE = collections.Counter()
_CACHE_ROLE_RAW_KEY_PRESENCE = collections.Counter()
_CACHE_INTEGRITY_UIDS = set()

for _uid in PROBE_UIDS:
    _paths = list(cache_paths_by_uid.get(str(_uid), []))
    if not _paths:
        _CACHE_COUNTS["missing_role_cache_uid"] += 1
        _CACHE_INTEGRITY_UIDS.add(str(_uid))
        continue
    if len(_paths) > 1:
        _CACHE_COPY_UIDS += 1
    _candidates = []
    for _path in _paths:
        try:
            with np.load(_path, allow_pickle=False) as _saved:
                _CACHE_KEY_NAMES.update(str(key) for key in _saved.files)
                _CACHE_COUNTS["npz_files_checked"] += 1
                record_field_presence(_CACHE_KEY_FILE_PRESENCE, _saved.files)
                _pose = np.asarray(_saved["pose"], dtype=np.float32).copy() if "pose" in _saved.files else None
                _conf = np.asarray(_saved["confidence"], dtype=np.float32).copy() if "confidence" in _saved.files else None
                if _pose is None or _conf is None:
                    _CACHE_COPY_INVALID += 1
                    _CACHE_INTEGRITY_UIDS.add(str(_uid))
                    continue
                if _pose.shape != (48, 1728) or _conf.shape != (48, 576):
                    _CACHE_COUNTS["raw_shape_mismatch_files"] += 1
                    _CACHE_COPY_INVALID += 1
                    _CACHE_INTEGRITY_UIDS.add(str(_uid))
                    continue
                _raw_count = None
                _fps = None
                _timestamp = None
                if "raw_frame_count" in _saved.files:
                    _raw_count = int(np.asarray(_saved["raw_frame_count"]).reshape(-1)[0])
                for _fps_key in ("fps", "frame_rate"):
                    if _fps_key in _saved.files:
                        _fps = float(np.asarray(_saved[_fps_key]).reshape(-1)[0])
                        break
                for _timestamp_key in _TIMESTAMP_KEYS:
                    if _timestamp_key in _saved.files:
                        _timestamp = np.asarray(_saved[_timestamp_key], dtype=np.float64).reshape(-1).copy()
                        break
                _present_windows = [key for key in _WINDOW_KEYS if key in _saved.files]
                _present_mirror = [key for key in _MIRROR_KEYS if key in _saved.files]
                _present_keys = tuple(str(key) for key in _saved.files)
                _candidates.append((_pose, _conf, _raw_count, _fps, _timestamp, _present_windows, _present_mirror, _present_keys))
        except Exception:
            _CACHE_COUNTS["unreadable_npz_files"] += 1
            _CACHE_COPY_INVALID += 1
            _CACHE_INTEGRITY_UIDS.add(str(_uid))
    if not _candidates:
        _CACHE_COUNTS["role_uids_without_valid_pose_cache"] += 1
        _CACHE_INTEGRITY_UIDS.add(str(_uid))
        continue
    _reference = _candidates[0]
    for _candidate in _candidates[1:]:
        if set(_candidate[7]) != set(_reference[7]):
            _CACHE_COPY_KEYSET_CONFLICT_UIDS.add(str(_uid))
        _same_metadata = cache_metadata_copies_match(_candidate, _reference)
        if not np.array_equal(_candidate[0], _reference[0]) or not np.array_equal(_candidate[1], _reference[1]) or not _same_metadata:
            _CACHE_COPY_CONFLICT_UIDS.add(str(_uid))
            _CACHE_INTEGRITY_UIDS.add(str(_uid))
    _CACHE_COUNTS["role_uids_with_valid_pose_cache"] += 1
    _CACHE_COUNTS["canonical_48_frame_pose_shapes"] += int(_reference[0].shape == (48, 1728))
    _CACHE_COUNTS["canonical_48_frame_confidence_shapes"] += int(_reference[1].shape == (48, 576))
    _ROLE_KEY_UNION = set().union(*(set(candidate[7]) for candidate in _candidates))
    record_field_presence(_CACHE_ROLE_RAW_KEY_PRESENCE, _ROLE_KEY_UNION)
    _ROLE_GROUP_FIELDS = []
    if "pose" in _ROLE_KEY_UNION:
        _ROLE_GROUP_FIELDS.append("pose")
    if "confidence" in _ROLE_KEY_UNION:
        _ROLE_GROUP_FIELDS.append("confidence")
    if "raw_frame_count" in _ROLE_KEY_UNION:
        _ROLE_GROUP_FIELDS.append("raw_frame_count")
    if {"fps", "frame_rate"} & _ROLE_KEY_UNION:
        _ROLE_GROUP_FIELDS.append("fps_or_frame_rate")
    if set(_TIMESTAMP_KEYS) & _ROLE_KEY_UNION:
        _ROLE_GROUP_FIELDS.append("timestamps")
    if set(_WINDOW_KEYS) & _ROLE_KEY_UNION:
        _ROLE_GROUP_FIELDS.append("sequence_window")
    if set(_MIRROR_KEYS) & _ROLE_KEY_UNION:
        _ROLE_GROUP_FIELDS.append("mirror_or_handedness")
    if "handedness" in _ROLE_KEY_UNION:
        _ROLE_GROUP_FIELDS.append("handedness")
    record_field_presence(_CACHE_ROLE_KEY_PRESENCE, _ROLE_GROUP_FIELDS)
    _all_raw_counts = [candidate[2] for candidate in _candidates if candidate[2] is not None]
    _all_fps_values = [candidate[3] for candidate in _candidates if candidate[3] is not None]
    _timestamp_candidates = [candidate for candidate in _candidates if candidate[4] is not None]
    _present_windows = set().union(*(set(candidate[5]) for candidate in _candidates))
    _present_mirror = set().union(*(set(candidate[6]) for candidate in _candidates))
    if _all_raw_counts:
        _CACHE_COUNTS["raw_frame_count_available_rows"] += 1
        if any(int(value) < 2 for value in _all_raw_counts):
            _CACHE_COUNTS["invalid_raw_frame_count_rows"] += 1
    if _all_fps_values:
        _CACHE_COUNTS["fps_available_rows"] += 1
        if any(not np.isfinite(value) or float(value) <= 0 for value in _all_fps_values):
            _CACHE_COUNTS["invalid_fps_rows"] += 1
    if _timestamp_candidates:
        _TIMESTAMP_PRESENT += 1
        _checks = [
            timestamp_grid_check(candidate[4], raw_frame_count=candidate[2], fps=candidate[3], expected_frame_count=48)
            for candidate in _timestamp_candidates
        ]
        _statuses = {str(check["status"]) for check in _checks}
        if "failed" in _statuses:
            _TIMESTAMP_CHECKS["failed"] += 1
        elif "invalid_metadata" in _statuses:
            _TIMESTAMP_CHECKS["invalid_metadata"] += 1
        elif "passed" in _statuses:
            _TIMESTAMP_CHECKS["passed"] += 1
        else:
            _TIMESTAMP_CHECKS["unavailable"] += 1
    else:
        _TIMESTAMP_CHECKS["unavailable"] += 1
    if _present_windows:
        _WINDOW_PRESENT += 1
    if _present_mirror:
        _MIRROR_PRESENT += 1
    if "handedness" in _present_mirror:
        _HANDEDNESS_PRESENT += 1
_PROBE_ROLE_ROWS = len(PROBE_UIDS)
_CACHE_COUNTS["expected_fit_and_inner_stop_pose_rows"] = int(_PROBE_ROLE_ROWS)
if _CACHE_COUNTS["role_uids_with_valid_pose_cache"] != _PROBE_ROLE_ROWS:
    raise RuntimeError("Pose cache coverage is incomplete for the fixed fit/inner-stop roles; no audit artifact was written.")

# Capture only metadata key presence, never key values or row paths.
_CACHE_KEY_NAMES = sorted(_CACHE_KEY_NAMES)
_CACHE_METADATA_KEY_NAMES = sorted(set(_CACHE_KEY_NAMES) | {
    "pose", "confidence", "raw_frame_count", "fps", "frame_rate",
    *_TIMESTAMP_KEYS, *_WINDOW_KEYS, *_MIRROR_KEYS,
})
_SOURCE_UID_MISMATCHES = [row for row in ALL_ROLE_RECORDS if bool(row.get("source_uid_mismatch", False))]
_EXPLAINED_SOURCE_MISMATCHES = sum(
    "official" in str(row.get("source_video_id_provenance", "")).lower()
    for row in _SOURCE_UID_MISMATCHES
)
_UNEXPLAINED_SOURCE_MISMATCHES = len(_SOURCE_UID_MISMATCHES) - _EXPLAINED_SOURCE_MISMATCHES
_SOURCE_PROVENANCE_COUNTS = dict(collections.Counter(
    str(row.get("source_video_id_provenance", "unavailable")) for row in ALL_ROLE_RECORDS
))

# Rebuild the previous support probe's fixed cohort exactly: same caption
# vectors, query shuffle seed, shared confidence gate, text/source exclusions,
# semantic threshold, and minimum common-donor count. This performs no fitting.
PROBE_RAW_CAPTIONS = sorted(
    {str(row["text"]) for row in PROBE_RECORDS},
    key=lambda value: hashlib.sha256(value.encode("utf-8")).hexdigest(),
)
RAW_CAPTION_TO_INDEX = {caption: index for index, caption in enumerate(PROBE_RAW_CAPTIONS)}
PROBE_TEXT_INDEX = np.asarray([RAW_CAPTION_TO_INDEX[str(row["text"])] for row in PROBE_RECORDS], dtype=np.int64)
_encoder = AutoModel.from_pretrained(TEXT_ENCODER_NAME, revision=TEXT_ENCODER_REVISION)
_tokenizer = AutoTokenizer.from_pretrained(TEXT_ENCODER_NAME, revision=TEXT_ENCODER_REVISION)
ACTUAL_TEXT_ENCODER_REVISION = str(getattr(_encoder.config, "_commit_hash", "") or "")
assert ACTUAL_TEXT_ENCODER_REVISION == TEXT_ENCODER_REVISION
_encoder_hash = hashlib.sha256()
for _name, _tensor in sorted(_encoder.state_dict().items()):
    _encoder_hash.update(_name.encode("utf-8"))
    _encoder_hash.update(_tensor.detach().cpu().contiguous().numpy().tobytes(order="C"))
TEXT_ENCODER_STATE_SHA256 = _encoder_hash.hexdigest().upper()
TOKENIZER_VOCAB_SHA256 = hashlib.sha256(
    json.dumps(_tokenizer.get_vocab(), sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest().upper()
_encoder = _encoder.to(device=device).eval()
_raw_text_embeddings = []
with torch.no_grad():
    for _start in range(0, len(PROBE_RAW_CAPTIONS), 64):
        _batch = _tokenizer(
            PROBE_RAW_CAPTIONS[_start:_start + 64], max_length=128, padding=True,
            truncation=True, return_tensors="pt",
        ).to(device)
        _hidden = _encoder(**_batch).last_hidden_state
        _mask = _batch["attention_mask"].unsqueeze(-1).to(_hidden.dtype)
        _pooled = (_hidden * _mask).sum(dim=1) / _mask.sum(dim=1).clamp_min(1.0)
        _raw_text_embeddings.append(torch.nn.functional.normalize(_pooled.float(), p=2, dim=-1).cpu().numpy())
PROBE_TEXT_UNIT = np.concatenate(_raw_text_embeddings, axis=0).astype(np.float32)
assert PROBE_TEXT_UNIT.shape[0] == len(PROBE_RAW_CAPTIONS) and np.isfinite(PROBE_TEXT_UNIT).all()
del _raw_text_embeddings, _hidden, _mask, _pooled, _batch, _encoder
torch.cuda.empty_cache()
gc.collect()
DONOR_TEXT = PROBE_TEXT_UNIT[PROBE_TEXT_INDEX[:len(fit_uids)]]
QUERY_TEXT = PROBE_TEXT_UNIT[PROBE_TEXT_INDEX[len(fit_uids):]]
QUERY_CAPTION_KEYS = [str(row.get("normalized_caption_key") or normalize_caption(str(row["text"]))) for row in STOP_RECORDS]
DONOR_CAPTION_KEYS = [str(row.get("normalized_caption_key") or normalize_caption(str(row["text"]))) for row in FIT_RECORDS]
DONOR_ALIASES = [aliases_for_record(row) for row in FIT_RECORDS]
QUERY_ALIASES = [aliases_for_record(row) for row in STOP_RECORDS]
QUERY_SHUFFLE_INDEX = source_group_derangement(
    QUERY_ALIAS_GROUPS, QUERY_ALIAS_GROUPS, QUERY_CAPTION_KEYS, QUERY_CAPTION_KEYS,
    QUERY_TEXT, threshold=SEMANTIC_DUPLICATE_COSINE_THRESHOLD, seed=QUERY_SHUFFLE_SEED,
)
QUERY_SHUFFLED_ALIASES = [QUERY_ALIASES[int(index)] for index in QUERY_SHUFFLE_INDEX]

# Support helpers are copied from the previously reviewed no-training probe.
DONOR_CONFIDENCE_ROWS = [np.asarray(canonical_conf_by_uid[uid], dtype=np.float32) for uid in fit_uids]
QUERY_CONFIDENCE_ROWS = [np.asarray(canonical_conf_by_uid[uid], dtype=np.float32) for uid in early_uids]
DONOR_SUPPORT_BITS = stack_confidence_support_rows(DONOR_CONFIDENCE_ROWS)
COMMON_MASKS = []
SCORABLE_QUERY_INDICES = []
_COMMON_COUNTS = collections.Counter()
for _query_ix, _query_uid in enumerate(early_uids):
    _, _all_group_support, _ = paired_group_support(
        QUERY_CONFIDENCE_ROWS[_query_ix], DONOR_SUPPORT_BITS,
    )
    _shuffled_ix = int(QUERY_SHUFFLE_INDEX[_query_ix])
    _common_mask, _filter_counts = common_donor_mask(
        QUERY_ALIASES[_query_ix], QUERY_ALIASES[_query_ix], QUERY_SHUFFLED_ALIASES[_query_ix],
        QUERY_CAPTION_KEYS[_query_ix], QUERY_CAPTION_KEYS[_shuffled_ix],
        DONOR_ALIASES, DONOR_CAPTION_KEYS, DONOR_TEXT,
        QUERY_TEXT[_query_ix], QUERY_TEXT[_shuffled_ix], _all_group_support,
        semantic_threshold=SEMANTIC_DUPLICATE_COSINE_THRESHOLD,
    )
    _COMMON_COUNTS["candidate_donor_pairs"] += int(_filter_counts["candidate_donors"])
    _COMMON_COUNTS["confidence_supported_common_donor_pairs"] += int(_filter_counts["common_donors"])
    if int(_filter_counts["common_donors"]) >= MIN_COMMON_DONORS_PER_QUERY:
        SCORABLE_QUERY_INDICES.append(_query_ix)
        COMMON_MASKS.append(_common_mask)
    else:
        COMMON_MASKS.append(None)
assert len(SCORABLE_QUERY_INDICES) == EXPECTED_FIXED_QUERY_COUNT, \
    "Reconstructed fixed common-support cohort differs from the reviewed 985-query cohort."
assert len(set(SCORABLE_QUERY_INDICES)) == EXPECTED_FIXED_QUERY_COUNT
SCORABLE_QUERY_CONFIDENCE = np.stack([QUERY_CONFIDENCE_ROWS[index] for index in SCORABLE_QUERY_INDICES])
FIXED_COHORT_SUPPORT = confidence_support_report(
    SCORABLE_QUERY_CONFIDENCE,
    {name: (start, end) for name, (start, end, _) in GROUPS.items()},
    threshold=MIN_CONFIDENCE_FOR_SUPPORT,
)

# Embed unique normalized caption keys (all roles) for the separate duplicate
# audit. This includes external development captions for leak counting only;
# their pose arrays are never read. Common-cohort embeddings above retain the
# exact prior fit/inner-stop caption set and batching order.
ALL_CAPTION_KEYS = sorted({normalize_caption(str(row["text"])) for row in ALL_ROLE_RECORDS})
_near_text_encoder = AutoModel.from_pretrained(TEXT_ENCODER_NAME, revision=TEXT_ENCODER_REVISION)
_near_tokenizer = AutoTokenizer.from_pretrained(TEXT_ENCODER_NAME, revision=TEXT_ENCODER_REVISION)
assert str(getattr(_near_text_encoder.config, "_commit_hash", "") or "") == TEXT_ENCODER_REVISION
_near_text_encoder = _near_text_encoder.to(device=device).eval()
_near_embeddings = []
with torch.no_grad():
    for _start in range(0, len(ALL_CAPTION_KEYS), 64):
        _batch = _near_tokenizer(
            ALL_CAPTION_KEYS[_start:_start + 64], max_length=128, padding=True,
            truncation=True, return_tensors="pt",
        ).to(device)
        _hidden = _near_text_encoder(**_batch).last_hidden_state
        _mask = _batch["attention_mask"].unsqueeze(-1).to(_hidden.dtype)
        _pooled = (_hidden * _mask).sum(dim=1) / _mask.sum(dim=1).clamp_min(1.0)
        _near_embeddings.append(torch.nn.functional.normalize(_pooled.float(), p=2, dim=-1).cpu().numpy())
CAPTION_KEY_EMBEDDINGS = np.concatenate(_near_embeddings, axis=0).astype(np.float32)
del _near_embeddings, _hidden, _mask, _pooled, _batch, _near_text_encoder
torch.cuda.empty_cache()
gc.collect()
CAPTION_KEY_INDEX = {key: index for index, key in enumerate(ALL_CAPTION_KEYS)}
NEAR_COMPONENT_LABELS, NEAR_CAPTION_EDGE_INDICES, NEAR_DUPLICATE_DIAGNOSTICS = caption_near_duplicate_components(
    CAPTION_KEY_EMBEDDINGS,
    threshold=SEMANTIC_DUPLICATE_COSINE_THRESHOLD,
    chunk_size=256,
)

ROLE_NAME_BY_ROW = ALL_ROLE_NAMES
ALIAS_GROUP_BY_ROW = ALL_ALIAS_GROUPS
CAPTION_KEY_BY_ROW = [normalize_caption(str(row["text"])) for row in ALL_ROLE_RECORDS]
ROWS_BY_CAPTION_KEY = collections.defaultdict(list)
ROLE_SOURCE_COUNTS_BY_CAPTION_KEY = collections.defaultdict(collections.Counter)
for _row_ix, (_key, _role, _source_group) in enumerate(zip(CAPTION_KEY_BY_ROW, ROLE_NAME_BY_ROW, ALIAS_GROUP_BY_ROW)):
    ROWS_BY_CAPTION_KEY[_key].append(_row_ix)
    ROLE_SOURCE_COUNTS_BY_CAPTION_KEY[_key][(_role, int(_source_group))] += 1

def _caption_pair_counts(left_key, right_key):
    left_counts = ROLE_SOURCE_COUNTS_BY_CAPTION_KEY[left_key]
    right_counts = ROLE_SOURCE_COUNTS_BY_CAPTION_KEY[right_key]
    cross_source = 0
    role_pairs = collections.Counter()
    for (left_role, left_group), left_count in left_counts.items():
        for (right_role, right_group), right_count in right_counts.items():
            pair_count = int(left_count * right_count)
            if left_group != right_group:
                cross_source += pair_count
            if left_role != right_role:
                role_pairs[(left_role, right_role)] += pair_count
    return cross_source, role_pairs

_EXACT_CLUSTERS = [rows for rows in ROWS_BY_CAPTION_KEY.values() if len(rows) > 1]
_EXACT_CROSS_SOURCE_PAIR_COUNT = 0
_EXACT_CROSS_SOURCE_CLUSTER_COUNT = 0
_EXACT_CROSS_SOURCE_ROW_COUNT = 0
for _key, _rows in ROWS_BY_CAPTION_KEY.items():
    _sources = {int(ALIAS_GROUP_BY_ROW[index]) for index in _rows}
    if len(_sources) > 1:
        _EXACT_CROSS_SOURCE_CLUSTER_COUNT += 1
        _EXACT_CROSS_SOURCE_ROW_COUNT += len(_rows)
        for _left, _right in combinations(_rows, 2):
            if ALIAS_GROUP_BY_ROW[_left] != ALIAS_GROUP_BY_ROW[_right]:
                _EXACT_CROSS_SOURCE_PAIR_COUNT += 1

_NEAR_CROSS_SOURCE_EDGE_COUNT = 0
_NEAR_CROSS_SOURCE_ROW_PAIR_COUNT = 0
_NEAR_CROSS_ROLE_EDGE_COUNTS = collections.Counter()
_NEAR_CROSS_ROLE_ROW_PAIR_COUNTS = collections.Counter()
for _left_ix, _right_ix in NEAR_CAPTION_EDGE_INDICES:
    _left_key, _right_key = ALL_CAPTION_KEYS[int(_left_ix)], ALL_CAPTION_KEYS[int(_right_ix)]
    _cross_source_pairs, _role_pairs = _caption_pair_counts(_left_key, _right_key)
    if _cross_source_pairs:
        _NEAR_CROSS_SOURCE_EDGE_COUNT += 1
        _NEAR_CROSS_SOURCE_ROW_PAIR_COUNT += _cross_source_pairs
    for _role_pair, _pair_count in _role_pairs.items():
        _sorted_role_pair = tuple(sorted(_role_pair))
        _NEAR_CROSS_ROLE_EDGE_COUNTS[_sorted_role_pair] += 1
        _NEAR_CROSS_ROLE_ROW_PAIR_COUNTS[_sorted_role_pair] += int(_pair_count)

_COMPONENT_SOURCES = collections.defaultdict(set)
_COMPONENT_ROWS = collections.Counter()
_COMPONENT_ROLES = collections.defaultdict(set)
for _key, _rows in ROWS_BY_CAPTION_KEY.items():
    _component = int(NEAR_COMPONENT_LABELS[CAPTION_KEY_INDEX[_key]])
    _COMPONENT_ROWS[_component] += len(_rows)
    for _row_ix in _rows:
        _COMPONENT_SOURCES[_component].add(int(ALIAS_GROUP_BY_ROW[_row_ix]))
        _COMPONENT_ROLES[_component].add(ROLE_NAME_BY_ROW[_row_ix])
_CROSS_SOURCE_COMPONENTS = [component for component, sources in _COMPONENT_SOURCES.items() if len(sources) > 1]
_CROSS_ROLE_COMPONENTS = [component for component, roles in _COMPONENT_ROLES.items() if len(roles) > 1]

# Motion variation summaries use only fit + inner-stop rows. Exact duplicate
# and direct near-caption pairs must come from different source/alias groups.
FIT_STOP_ROW_BY_GLOBAL = {index: local for local, index in enumerate(range(len(PROBE_RECORDS)))}
GLOBAL_TO_FIT_STOP = {global_index: local_index for local_index, global_index in FIT_STOP_ROW_BY_GLOBAL.items()}
FIT_STOP_GLOBAL_ROWS = list(range(len(PROBE_RECORDS)))
FIT_STOP_SOURCE_BY_ROW = FIT_STOP_ALIAS_GROUPS
FIT_STOP_POSES = np.stack([
    (np.asarray(canonical_pose_by_uid[uid], dtype=np.float32) - v5_mean[None]) / v5_std[None]
    for uid in PROBE_UIDS
])
FIT_STOP_CONFIDENCE = np.stack([np.asarray(canonical_conf_by_uid[uid], dtype=np.float32) for uid in PROBE_UIDS])
FIT_STOP_CAPTION_KEYS = [normalize_caption(str(row["text"])) for row in PROBE_RECORDS]
FIT_STOP_ROWS_BY_CAPTION = collections.defaultdict(list)
FIT_STOP_ROWS_BY_SOURCE = collections.defaultdict(list)
for _row_ix, (_key, _source) in enumerate(zip(FIT_STOP_CAPTION_KEYS, FIT_STOP_SOURCE_BY_ROW)):
    FIT_STOP_ROWS_BY_CAPTION[_key].append(_row_ix)
    FIT_STOP_ROWS_BY_SOURCE[int(_source)].append(_row_ix)

def _cross_source_blocks(left_rows, right_rows=None):
    if right_rows is None:
        grouped = collections.defaultdict(list)
        for index in left_rows:
            grouped[int(FIT_STOP_SOURCE_BY_ROW[index])].append(int(index))
        blocks = []
        groups = sorted(grouped)
        for left_group_ix, left_group in enumerate(groups):
            for right_group in groups[left_group_ix + 1:]:
                blocks.append(("cross", grouped[left_group], grouped[right_group]))
        return blocks
    left_grouped, right_grouped = collections.defaultdict(list), collections.defaultdict(list)
    for index in left_rows:
        left_grouped[int(FIT_STOP_SOURCE_BY_ROW[index])].append(int(index))
    for index in right_rows:
        right_grouped[int(FIT_STOP_SOURCE_BY_ROW[index])].append(int(index))
    return [
        ("cross", left_grouped[left_group], right_grouped[right_group])
        for left_group in sorted(left_grouped)
        for right_group in sorted(right_grouped)
        if left_group != right_group
    ]

def _pair_block_population(block):
    kind, left_rows, right_rows = block
    return len(left_rows) * (len(left_rows) - 1) // 2 if kind == "same" else len(left_rows) * len(right_rows)

def _sample_pair_blocks(blocks, cap, seed):
    population = int(sum(_pair_block_population(block) for block in blocks))
    if not population:
        return [], population, 0, "none"
    rng = np.random.default_rng(seed)
    if population <= cap:
        sampled = []
        for kind, left_rows, right_rows in blocks:
            if kind == "same":
                sampled.extend((left, right) for left, right in combinations(left_rows, 2))
            else:
                sampled.extend((left, right) for left in left_rows for right in right_rows)
        return sampled, population, population, "complete_enumeration"
    weights = np.asarray([_pair_block_population(block) for block in blocks], dtype=np.float64)
    probabilities = weights / weights.sum()
    chosen_blocks = rng.choice(len(blocks), size=int(cap), replace=True, p=probabilities)
    sampled = []
    for block_ix in chosen_blocks:
        kind, left_rows, right_rows = blocks[int(block_ix)]
        if kind == "same":
            offsets = rng.choice(len(left_rows), size=2, replace=False)
            sampled.append((left_rows[int(offsets[0])], left_rows[int(offsets[1])]))
        else:
            sampled.append((left_rows[int(rng.integers(len(left_rows)))], right_rows[int(rng.integers(len(right_rows)))]))
    return sampled, population, len(sampled), "uniform_weighted_sampling_with_replacement"

def _motion_variation(blocks, seed):
    pairs, population, sample_count, scheme = _sample_pair_blocks(blocks, 25000, seed)
    distances = []
    unsupported = 0
    for left_ix, right_ix in pairs:
        value = confidence_weighted_pair_mse(
            FIT_STOP_POSES[left_ix], FIT_STOP_CONFIDENCE[left_ix],
            FIT_STOP_POSES[right_ix], FIT_STOP_CONFIDENCE[right_ix],
            threshold=MIN_CONFIDENCE_FOR_SUPPORT,
        )
        if value is None:
            unsupported += 1
        else:
            distances.append(value)
    summary = paired_distance_summary(distances)
    summary.update({
        "candidate_pair_count": int(population),
        "sampled_pair_count": int(sample_count),
        "unsupported_sampled_pair_count": int(unsupported),
        "sampling_scheme": scheme,
        "metric": "confidence-weighted canonical V5-normalized XYZ MSE over shared supported cells",
    })
    return summary

_WITHIN_SOURCE_BLOCKS = []
for _rows in FIT_STOP_ROWS_BY_SOURCE.values():
    if len(_rows) > 1:
        _WITHIN_SOURCE_BLOCKS.append(("same", _rows, _rows))
_EXACT_CROSS_SOURCE_BLOCKS = []
for _rows in FIT_STOP_ROWS_BY_CAPTION.values():
    _EXACT_CROSS_SOURCE_BLOCKS.extend(_cross_source_blocks(_rows))
_NEAR_CROSS_SOURCE_BLOCKS = []
for _left_ix, _right_ix in NEAR_CAPTION_EDGE_INDICES:
    _left_key, _right_key = ALL_CAPTION_KEYS[int(_left_ix)], ALL_CAPTION_KEYS[int(_right_ix)]
    _left_rows = [index for index in FIT_STOP_ROWS_BY_CAPTION.get(_left_key, [])]
    _right_rows = [index for index in FIT_STOP_ROWS_BY_CAPTION.get(_right_key, [])]
    _NEAR_CROSS_SOURCE_BLOCKS.extend(_cross_source_blocks(_left_rows, _right_rows))

MOTION_DIVERSITY = {
    "within_source_repeat_variation": _motion_variation(_WITHIN_SOURCE_BLOCKS, 8301),
    "cross_source_exact_caption_variation": _motion_variation(_EXACT_CROSS_SOURCE_BLOCKS, 8302),
    "cross_source_direct_near_caption_variation": _motion_variation(_NEAR_CROSS_SOURCE_BLOCKS, 8303),
}

# Processing consistency and missing-metadata accounting. The original V2
# cache is already resampled and does not contain raw-video timestamps.
_EXPECTED_POSE_SHAPE = (48, 1728)
_EXPECTED_CONFIDENCE_SHAPE = (48, 576)
_CACHE_METADATA_REPORT = {
    "role_pose_rows_checked": int(_PROBE_ROLE_ROWS),
    "role_rows_with_valid_cache": int(_CACHE_COUNTS["role_uids_with_valid_pose_cache"]),
    "pose_shape_valid_files": int(_CACHE_COUNTS["canonical_48_frame_pose_shapes"]),
    "confidence_shape_valid_files": int(_CACHE_COUNTS["canonical_48_frame_confidence_shapes"]),
    "missing_role_cache_uids": int(_CACHE_COUNTS["missing_role_cache_uid"] + _CACHE_COUNTS["role_uids_without_valid_pose_cache"]),
    "invalid_or_unreadable_cache_copies": int(_CACHE_COPY_INVALID),
    "duplicate_cache_copy_uid_count": int(_CACHE_COPY_UIDS),
    "conflicting_duplicate_cache_uid_count": int(len(_CACHE_COPY_CONFLICT_UIDS)),
    "duplicate_cache_keyset_mismatch_uid_count": int(len(_CACHE_COPY_KEYSET_CONFLICT_UIDS)),
    "role_uid_key_presence_rule": "per-role-UID key presence is the union across its valid cache copies; exact key-set disagreements are reported separately",
    "npz_file_count_checked": int(_CACHE_COUNTS["npz_files_checked"]),
    "observed_npz_key_names": _CACHE_KEY_NAMES,
    "npz_key_presence_by_file": field_presence_report(
        _CACHE_KEY_FILE_PRESENCE,
        int(_CACHE_COUNTS["npz_files_checked"]),
        _CACHE_METADATA_KEY_NAMES,
        denominator_name="readable_npz_file_denominator",
        present_name="present_file_count",
        missing_name="missing_file_count",
    ),
    "npz_key_presence_by_valid_role_uid": field_presence_report(
        _CACHE_ROLE_RAW_KEY_PRESENCE,
        int(_CACHE_COUNTS["role_uids_with_valid_pose_cache"]),
        _CACHE_METADATA_KEY_NAMES,
        denominator_name="valid_role_uid_denominator",
        present_name="present_uid_count",
        missing_name="missing_uid_count",
    ),
    "known_metadata_field_presence_by_role_uid": field_presence_report(
        _CACHE_ROLE_KEY_PRESENCE,
        int(_CACHE_COUNTS["role_uids_with_valid_pose_cache"]),
        ("pose", "confidence", "raw_frame_count", "fps_or_frame_rate", "timestamps", "sequence_window", "mirror_or_handedness", "handedness"),
        denominator_name="valid_role_uid_denominator",
        present_name="present_uid_count",
        missing_name="missing_uid_count",
    ),
    "raw_frame_count_available_rows": int(_CACHE_COUNTS["raw_frame_count_available_rows"]),
    "raw_frame_count_invalid_rows": int(_CACHE_COUNTS["invalid_raw_frame_count_rows"]),
    "fps_available_rows": int(_CACHE_COUNTS["fps_available_rows"]),
    "fps_invalid_rows": int(_CACHE_COUNTS["invalid_fps_rows"]),
    "timestamp_available_rows": int(_TIMESTAMP_PRESENT),
    "sequence_window_metadata_available_rows": int(_WINDOW_PRESENT),
    "mirror_or_handedness_metadata_available_rows": int(_MIRROR_PRESENT),
    "handedness_metadata_available_rows": int(_HANDEDNESS_PRESENT),
    "timestamp_half_frame_check": {
        "tolerance_definition": "one half of a source frame duration (0.5/fps)",
        "checked_rows": int(_TIMESTAMP_CHECKS["passed"] + _TIMESTAMP_CHECKS["failed"]),
        "passed_rows": int(_TIMESTAMP_CHECKS["passed"]),
        "failed_rows": int(_TIMESTAMP_CHECKS["failed"]),
        "unavailable_rows": int(_TIMESTAMP_CHECKS["unavailable"]),
        "invalid_metadata_rows": int(_TIMESTAMP_CHECKS["invalid_metadata"]),
    },
    "resampling_policy_declared": str(manifest.get("resampling", {}).get("policy", "unavailable")),
    "resampled_frame_count_declared": int(manifest.get("resampling", {}).get("resampled_frame_count", -1)),
    "resampling_code_contract": "linear interpolation at 48 uniformly spaced frame-index positions including both endpoints",
    "raw_v2_frame_count_documented": "unavailable for the legacy V2 cached pose arrays",
    "fps_timestamps_and_source_window": "report available cache metadata only; never inferred",
    "coordinate_space": str(manifest.get("coordinate_space", "unavailable")),
    "topology_matches_body_left_and_right_hands": True,
    "canonicalizer_mirror_or_axis_flip_operation": False,
    "canonicalizer_left_right_swap_operation": False,
    "canonicalizer_source_sha256": CANONICALIZER_SOURCE_SHA256,
    "canonicalizer_transform_contract": "builder-verified ordered point selection with nose/shoulder centering and positive x/y-derived scale applied uniformly; no mirror, axis-flip or left/right reorder operation in the embedded function",
    "mirror_and_handedness_metadata": "unavailable unless found in cache NPZ keys",
    "source_video_mirroring_or_handedness": "unavailable unless source metadata is present; canonicalizer code audit describes only transformations applied by this code",
    "word_level_timestamps": "unavailable; no word-level timing inferred",
}

_ACTIVE_ROLE_LEAKAGE_KEYS = (
    "fit_inner_stop_shared_uid_count", "fit_inner_stop_shared_alias_value_count",
    "fit_inner_stop_shared_exact_caption_key_count", "fit_external_dev_shared_uid_count",
    "fit_external_dev_shared_alias_value_count", "fit_external_dev_shared_exact_caption_key_count",
    "inner_stop_external_dev_shared_uid_count", "inner_stop_external_dev_shared_alias_value_count",
    "inner_stop_external_dev_shared_exact_caption_key_count",
)
_UNEXPLAINED_INTEGRITY_COUNT = int(
    CAPTION_KEY_MISMATCH_ROWS
    + _UNEXPLAINED_SOURCE_MISMATCHES
    + len(_CACHE_INTEGRITY_UIDS)
    + sum(ACTIVE_ROLE_INTEGRITY[key] for key in _ACTIVE_ROLE_LEAKAGE_KEYS)
)
_STATUS = "integrity_clean_with_metadata_gaps" if _UNEXPLAINED_INTEGRITY_COUNT == 0 else "review_required_unexplained_pairing_or_role_mismatch"
if _TIMESTAMP_CHECKS["failed"] or _TIMESTAMP_CHECKS["invalid_metadata"]:
    _STATUS = "review_required_timestamp_mismatch"

_REPORT = {
    "schema_version": 1,
    "audit": "private caption-pose-source/alias pairing and reference-target support audit",
    "status": _STATUS,
    "working_model": False,
    "signer_validated": False,
    "roles": {
        "fit_rows": int(len(fit_uids)),
        "inner_stop_query_rows_development_only": int(len(early_uids)),
        "fixed_common_support_query_rows": int(len(SCORABLE_QUERY_INDICES)),
        "unassigned_train_exclusion": UNASSIGNED_EXCLUSION_AUDIT,
        "reused_498_dev_metadata_rows_in_role_and_caption_duplicate_audits": int(len(EXTERNAL_DEV_RECORDS)),
        "reused_498_dev_pose_rows_loaded_or_scored": False,
        "reused_498_dev_pose_metrics_or_support_strata_computed": False,
        "reused_498_dev_results_used_for_model_tuning_or_evaluation": False,
        "role_integrity": ROLE_AUDIT,
        "active_role_overlap_counts": ACTIVE_ROLE_INTEGRITY,
    },
    "linkage_integrity": {
        "fit_and_inner_stop_rows_expected": int(_PROBE_ROLE_ROWS),
        "fit_and_inner_stop_pose_cache_rows_found": int(_CACHE_COUNTS["role_uids_with_valid_pose_cache"]),
        "duplicate_uid_rows": int(sum(int(value["duplicate_uid_rows"]) for value in ROLE_AUDIT["roles"].values())),
        "caption_key_mismatch_rows": int(CAPTION_KEY_MISMATCH_ROWS),
        "source_uid_mismatch_rows": int(len(_SOURCE_UID_MISMATCHES)),
        "source_uid_mismatches_explained_by_official_metadata_override": int(_EXPLAINED_SOURCE_MISMATCHES),
        "unexplained_source_uid_mismatches": int(_UNEXPLAINED_SOURCE_MISMATCHES),
        "unsupported_source_identity_rows": int(sum(int(value["unsupported_source_identity_rows"]) for value in ROLE_AUDIT["roles"].values())),
        "source_id_provenance_counts": _SOURCE_PROVENANCE_COUNTS,
        "unexplained_integrity_count": int(_UNEXPLAINED_INTEGRITY_COUNT),
    },
    "inputs": {
        "v2_pose_cache_run_key": EXPECTED_V2_RUN_KEY,
        "v5_manifest_sha256": V5_MANIFEST_SHA256,
        "v5_checkpoint_sha256": V5_CHECKPOINT_SHA256,
        "text_encoder": TEXT_ENCODER_NAME,
        "pinned_text_encoder_revision": ACTUAL_TEXT_ENCODER_REVISION,
        "text_encoder_state_sha256": TEXT_ENCODER_STATE_SHA256,
        "tokenizer_vocabulary_sha256": TOKENIZER_VOCAB_SHA256,
        "canonicalizer_source_sha256": CANONICALIZER_SOURCE_SHA256,
        "audit_core_sha256": AUDIT_CORE_SHA256,
        "audit_builder_sha256": AUDIT_BUILDER_SHA256,
    },
    "sequence_coordinate_preprocessing": _CACHE_METADATA_REPORT,
    "caption_duplicate_audit": {
        "exact_caption_normalizer": CAPTION_NORMALIZER,
        "unique_normalized_caption_count_all_roles": int(len(ALL_CAPTION_KEYS)),
        "exact_duplicate_caption_cluster_count_all_roles": int(len(_EXACT_CLUSTERS)),
        "rows_in_exact_duplicate_caption_clusters": int(sum(len(rows) for rows in _EXACT_CLUSTERS)),
        "exact_clusters_spanning_multiple_source_alias_groups": int(_EXACT_CROSS_SOURCE_CLUSTER_COUNT),
        "rows_in_cross_source_exact_caption_clusters": int(_EXACT_CROSS_SOURCE_ROW_COUNT),
        "cross_source_exact_caption_row_pair_count": int(_EXACT_CROSS_SOURCE_PAIR_COUNT),
        "near_duplicate_encoder": TEXT_ENCODER_NAME,
        "near_duplicate_encoder_revision": ACTUAL_TEXT_ENCODER_REVISION,
        "near_duplicate_caption_embedding_text": "V5-normalized exact-caption keys",
        "near_duplicate_cosine_threshold": float(SEMANTIC_DUPLICATE_COSINE_THRESHOLD),
        "near_duplicate_components": NEAR_DUPLICATE_DIAGNOSTICS,
        "components_spanning_multiple_source_alias_groups": int(len(_CROSS_SOURCE_COMPONENTS)),
        "rows_in_cross_source_near_caption_components": int(sum(_COMPONENT_ROWS[ix] for ix in _CROSS_SOURCE_COMPONENTS)),
        "components_spanning_multiple_roles": int(len(_CROSS_ROLE_COMPONENTS)),
        "direct_near_caption_edges_spanning_source_groups": int(_NEAR_CROSS_SOURCE_EDGE_COUNT),
        "direct_near_caption_row_pairs_spanning_source_groups": int(_NEAR_CROSS_SOURCE_ROW_PAIR_COUNT),
        "direct_near_caption_edge_counts_by_cross_role_pair": {
            "__".join(key): int(value) for key, value in sorted(_NEAR_CROSS_ROLE_EDGE_COUNTS.items())
        },
        "direct_near_caption_row_pair_counts_by_cross_role_pair": {
            "__".join(key): int(value) for key, value in sorted(_NEAR_CROSS_ROLE_ROW_PAIR_COUNTS.items())
        },
        "external_498_rows_in_caption_only_audit": True,
        "external_498_pose_rows_loaded_or_scored": False,
    },
    "aggregate_motion_diversity": MOTION_DIVERSITY,
    "fixed_985_reference_support": {
        "cohort_reconstruction": {
            "query_role_rows": int(len(early_uids)),
            "scorable_fixed_query_rows": int(len(SCORABLE_QUERY_INDICES)),
            "minimum_common_fit_donors_per_query": int(MIN_COMMON_DONORS_PER_QUERY),
            "query_caption_shuffle_seed": int(QUERY_SHUFFLE_SEED),
            "semantic_caption_exclusion_threshold": float(SEMANTIC_DUPLICATE_COSINE_THRESHOLD),
            "same_donor_intersection_rule": "reconstructed using prior probe's source/alias, normalized exact/near-caption and three-region confidence support gates",
        },
        "support": FIXED_COHORT_SUPPORT,
        "cohort_filtering_after_selection": False,
    },
    "next_training_gate": {
        "unexplained_pairing_or_active_role_leakage_count": int(_UNEXPLAINED_INTEGRITY_COUNT),
        "near_caption_cross_role_overlap_requires_review": bool(_NEAR_CROSS_ROLE_EDGE_COUNTS),
        "support_limited_hand_time_bins": int(sum(
            value["support_status"] == "support_limited"
            for region in ("left_hand", "right_hand")
            for value in FIXED_COHORT_SUPPORT["time_bins"][region].values()
        )),
        "model_training_started": False,
        "proceed_to_model_training_without_review": False,
    },
    "data_boundary": {
        "exported_artifact": "aggregate JSON only",
        "raw_clips_row_ids_aliases_captions_poses_predictions_embeddings_or_checkpoints_exported": False,
        "raw_clips_opened": False,
        "gated_isign_data_redistributed": False,
        "reused_498_dev_pose_scoring": False,
        "reused_498_dev_pose_metrics_or_support_strata_computed": False,
        "reused_498_dev_results_used_for_model_tuning_or_evaluation": False,
        "word_level_timing_inferred": False,
    },
}
AUDIT_OUTPUT_PATH = WORK_ROOT / "isl_pairing_target_audit_aggregate.json"
AUDIT_OUTPUT_PATH.write_text(json.dumps(_REPORT, indent=2, sort_keys=True), encoding="utf-8")
print("Pairing/target audit status:", _REPORT["status"], "| aggregate bytes:", AUDIT_OUTPUT_PATH.stat().st_size)
print("Role counts:", json.dumps(_REPORT["roles"], sort_keys=True))
print("Linkage integrity counts:", json.dumps(_REPORT["linkage_integrity"], sort_keys=True))
print("Cache metadata coverage:", json.dumps({key: _CACHE_METADATA_REPORT[key] for key in (
    "role_pose_rows_checked", "role_rows_with_valid_cache", "raw_frame_count_available_rows",
    "fps_available_rows", "timestamp_available_rows", "sequence_window_metadata_available_rows",
    "mirror_or_handedness_metadata_available_rows", "timestamp_half_frame_check",
    "npz_key_presence_by_file", "npz_key_presence_by_valid_role_uid",
    "known_metadata_field_presence_by_role_uid",
)}, sort_keys=True))
print("Caption duplicate aggregates:", json.dumps(_REPORT["caption_duplicate_audit"], sort_keys=True))
print("Motion diversity aggregates:", json.dumps(MOTION_DIVERSITY, sort_keys=True))
print("Fixed 985 reference-only support:", json.dumps(FIXED_COHORT_SUPPORT, sort_keys=True))
print("Aggregate-only output: no row identifiers, aliases, caption text, pose arrays, embeddings, predictions or checkpoint material saved.")
